# NuSE-E ×4, bản v3: cải thiện trunk trong ràng buộc, so với SPANV2, DISP, PLKSR-Tiny

Notebook này gồm toàn bộ code của NuSE-E (thiết kế v2 trong `plan/NuSE-E_handoff` cộng các biến thể v3 dưới đây), các baseline đã thử nghiệm, và phần so sánh. Trên Kaggle chỉ cần thêm dataset DIV2K, gói checkpoint cũ và output phiên trước là chạy được.

## Kết quả phiên trước (v2) và điều rút ra

| LR average pool, DIV2K valid | PSNR | tham số | ms fp32 @256² (T4) |
|---|---|---|---|
| NuSE-E v2 K=2 / 5 / 9 / 15 | 27.40 / 27.92 / 28.04 / 28.07 | 57k / 94k / 113k / 132k | 3.8 / 6.1 / 7.2 / 8.5 |
| SPANV2 + NuSE k=8 | 28.04 | 139k | — |
| DISP + NuSE k=8 | 28.11 | 157k | 8.4 |

1. **Từ K=9 trở lên, sai số nằm gần hết ở phần dự đoán hệ số** (91% ở K=9, 100% ở K=15; phần cắt bớt K gần bằng 0). Muốn tăng chất lượng phải tăng sức của trunk dùng chung, không phải thêm K.
2. **Còn dư 25k tham số** so với DISP + NuSE (157k) và latency đang bằng nhau. Trong khoảng đó có thể thêm block, mở rộng kênh, hoặc tăng trường tiếp nhận.
3. **K=2 bị vỡ khối 4×4** vì 2 thành phần không đủ biểu diễn khối (oracle 29.7 dB). Biến thể `e80` bỏ ngân sách này.
4. **150 s/epoch của phiên trước là do nạp dữ liệu** (mỗi crop giải mã lại cả ảnh 2K). Phiên này giữ ảnh train trong RAM, nên mỗi run NuSE-E dự kiến còn 30–60 phút. Nhờ vậy một phiên 12 giờ train được 8–10 biến thể trên 2 GPU.

## Biến thể v3

Mọi biến thể dùng cùng quy tắc suy kiến trúc từ phổ PCA (ngân sách K từ mục tiêu năng lượng, chia block theo năng lượng phi tuyến, nhánh tuyến tính theo R²). Chỉ đổi các siêu tham số mà quy tắc không quyết định:

| Tên | Trunk | Khác với v2 | Lý do |
|---|---|---|---|
| `NuSE-E` (v2) | N=7, W=32, block thường | — | tham chiếu |
| `d8` | N=8, W=32 | +1 block (151k) | dùng phần tham số còn dư |
| `rep7` | N=7, W=32, RepConv | train 3×3 + 1×1 + I, gộp thành 3×3 khi suy luận | cùng chi phí suy luận với v2 |
| `w40` | N=5, W=40 | rộng hơn, nông hơn (148k) | ít lớp tuần tự hơn, GPU dùng tốt hơn |
| `rep8` | N=8, W=32, RepConv | | |
| `lk7` | N=7, W=32, depthwise 7×7 trong block | +11k tham số | trường tiếp nhận rộng hơn (PLKSR-Tiny đạt 28.49 nhờ kernel 13×13) |
| `e80` | như v2, mục tiêu năng lượng (0.8, 0.9, 1.0) | ngân sách K = {5, 9, 15} | bỏ K=2 vỡ khối; trọng số loss dồn cho K lớn hơn |
| `top` | như v2, alpha = (0, 0, 0, 1) | loss = MSE ảnh ở K lớn nhất | đo xem trọng số nhiều ngân sách (Định lý 4) có làm K lớn nhất thiệt không |
| `w36` | N=6, W=36 | | |
| `w40rep` | N=5, W=40, RepConv | | |

Gộp RepConv là biến đổi đại số chính xác (test T11 kiểm output trước và sau gộp bằng nhau). Depthwise 7×7 và RepConv là khối chung đã có trong tài liệu (RepVGG, ECB, ConvNeXt); phần mới của bài vẫn là lý thuyết PCA null space, bộ giải mã cố định, khớp dữ liệu chính xác và kiến trúc co giãn theo K.

## Ràng buộc cứng (kiểm tự động)

| Ràng buộc | Kiểm ở đâu |
|---|---|
| Áp dụng đúng lý thuyết: UᵀU = I, A(SR) = y với mọi trọng số, head khởi tạo = Wiener | bước 2 (T4, T5, T10) |
| Co giãn theo K: output ở K không phụ thuộc stage sau; mỗi ngân sách thêm ≥ 1 block | bước 2 (T7, T9) |
| Nhẹ: tham số ở K lớn nhất ≤ DISP + NuSE k=8 (156,856) | bước 2b (loại trước khi train) và bước 7 |
| Realtime: latency @256² fp32 ở K lớn nhất ≤ DISP + NuSE k=8 trên cùng GPU | bước 2b và bước 7 |
| Tham số, latency, PSNR tăng đơn điệu theo K; max\|A(SR) − y\| < 1e-5 trên DIV2K valid | bước 7 |

Biến thể vi phạm ở bước 2b không được train. Bước 7 in bảng ràng buộc cho mọi biến thể đã train; bản được chọn mà vi phạm sẽ có cảnh báo.

## Cách chọn bản tốt nhất

Pha A train mọi biến thể qua sàng trên LR average pool (thứ tự ưu tiên như bảng trên). Pha B train thêm `PHASE_B_TOP` biến thể tốt nhất của pha A trên LR bicubic. Bản được chọn (★) là bản có PSNR DIV2K valid ở K lớn nhất cao nhất trong số đã train; bản v2 luôn được giữ để đối chiếu.

DIV2K valid vừa dùng để chọn `best.pt`, vừa dùng để chọn biến thể, nên số trên đó lạc quan một chút. Số cho bài báo lấy trên Set5, Set14, B100, Urban100, Manga109 (Add Input các bộ này là notebook tự đánh giá).

**Mọi file notebook ghi ra đều nằm trong một thư mục `notebook/`** (trên Kaggle là `/kaggle/working/notebook/`):

| Thư mục con | Nội dung |
|---|---|
| `code/` | `nuse_lib.py` (toàn bộ thư viện) và `train_run.py` (train một run), được ghi từ notebook này |
| `stats/<lr>/` | Thống kê PCA null space (`stats.pt`), kiến trúc suy ra cho v2 (`arch.json`) và từng biến thể (`arch_<tên>.json`) |
| `runs/<lr>/<model>/` | `train.log`, `history.json`, `last.pt`, `best.pt`, checkpoint mỗi 10 epoch; biến thể ở `nuse_e_<tên>` |
| `comparison/` | `screen.csv` (sàng), `variants.csv`, `constraints.csv`, `cost.csv`, `latency.csv`, `quality_by_config.csv`, hình |
| `comparison/<lr>/` | `summary.json`, `quality.csv`, `per_image.csv` |
| `notebook_outputs.zip` | Gói toàn bộ thư mục trên, trừ checkpoint trung gian |

`<lr>` là `average_pool` hoặc `bicubic`. Cache ảnh (file uint8 phẳng của tập train, BMP của tập valid) nằm ngoài thư mục này (`/tmp/nuse_cache` trên Kaggle).

## Cách chạy trên Kaggle

1. **Add Input** dataset DIV2K có `DIV2K_train_HR` (800 ảnh) và `DIV2K_valid_HR` (100 ảnh), ví dụ `soumikrakshit/div2k-high-resolution-images`. Notebook tự tìm hai thư mục này.
2. **Add Input** `nuse_old_runs_kaggle.zip` (57 MB, `best.pt` của 6 run cũ) đã upload thành dataset. Nhờ đó trần latency được đo lại trên cùng GPU và các baseline được đo lại đầy đủ (FID, Y-PSNR, từng ảnh). Thiếu file này thì bảng average pool dùng số đã báo cáo `[số cũ]` và trần latency dùng số cũ 8.44 ms.
3. **Add Input** output của phiên trước (thư mục `notebook/` có `runs/` và `stats/`). Notebook tự tìm (`RESUME_FROM = "auto"`), chép sang và bỏ qua mọi run đã có file `DONE`: bản v2 không phải train lại.
4. Nên có: các bộ benchmark Set5, Set14, B100, Urban100, Manga109 (thư mục chỉ chứa ảnh HR).
5. Settings: GPU T4 ×2; bật Internet để cài `lpips`, `torchmetrics` và tải trọng số AlexNet/Inception.
6. **Save & Run All (Commit).** Theo dõi ở tab Logs của version đang chạy.
   - Một phiên Kaggle tối đa 12 giờ. Notebook dừng nhận run mới khi sắp hết giờ (`SESSION_HOURS`); các bước đánh giá vẫn chạy với những run đã xong.
   - Phiên sau: Add Input output phiên này rồi Run All lại; run dở chạy tiếp từ `last.pt`.

Thời gian ước tính (T4 ×2, fp32, 100 epoch × 125 iteration): mỗi run NuSE-E 30–60 phút nếu loader RAM đạt kỳ vọng (log in `chờ dữ liệu` mỗi epoch để kiểm). 9 biến thể pha A + 2 run pha B ≈ 4–6 giờ; đánh giá và latency ≈ 1 giờ.

## So sánh công bằng

Mọi model (NuSE-E và các baseline) dùng chung các điều kiện sau. Chỉ khác ở kiến trúc và loss.

| Hạng mục | Điều kiện chung |
|---|---|
| Dữ liệu train | DIV2K train 800 ảnh, crop HR 256 / LR 64, xoay/lật, `repeat` 20, seed 2026 |
| Tạo LR | Cùng hàm cho mọi model |
| Lịch train | 100 epoch × 125 iteration, batch 128, AdamW lr 1e-3, cosine theo epoch đến 1e-6, grad clip 1.0 |
| Loss | Baseline: MSE ảnh. NuSE-E: MSE hệ số có trọng số, tương đương tổng MSE ảnh của mọi K (Định lý 4); với bicubic là tổng MSE ảnh (Định lý 7) |
| Đánh giá | DIV2K valid 100 ảnh toàn khung, cùng code metric |
| Latency | Cùng GPU, cùng cách đo |

Đây chính là giao thức của các run DISP, PLKSR-Tiny, SPANV2 trước đây, nên checkpoint LR average pool cũ dùng lại được; notebook kiểm tra lại cấu hình của chúng.

Với LR bicubic chưa có baseline nào được train (mỗi run baseline mất 4–10 giờ T4), nên bảng bicubic chỉ so NuSE-E v2 với các biến thể v3 và với nội suy bicubic.

NuSE-E được so ở K lớn nhất (chi phí gần SPANV2) và ở mọi K (đường chất lượng theo chi phí).

## Có cần train trên nhiều bộ dữ liệu không?

**Không cần cho giai đoạn này.**

- Các baseline cũ đều train trên DIV2K. Muốn so sánh công bằng thì NuSE-E cũng phải train đúng trên DIV2K với cùng lịch.
- Thứ cần nhiều bộ là **đánh giá**: DIV2K valid và năm bộ benchmark chuẩn (Set5, Set14, B100, Urban100, Manga109). Notebook tự đánh giá trên các bộ này nếu bạn attach.
- Train thêm DF2K (DIV2K + Flickr2K) hoặc LSDIR chỉ cần ở bản cuối, khi muốn đặt số liệu cạnh các bài SOTA. Khi đó mọi model so sánh phải được train lại trên cùng bộ đó.

In [ ]:
# ============================== Cấu hình ==============================
import os, sys, json, time, shutil, subprocess, statistics
from pathlib import Path

SMOKE = os.environ.get("NUSE_SMOKE") == "1"          # chạy thử cực ngắn (chỉ dùng khi kiểm tra code)
ON_KAGGLE = Path("/kaggle/input").is_dir()
INPUT_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path(os.environ.get("NUSE_INPUT", ".")).resolve()
# Local: nếu mở notebook ngay trong thư mục `notebook/` thì output ghi vào chính thư mục đó
WORK_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get(
    "NUSE_WORK", Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd())).resolve()
OUT = WORK_ROOT / "notebook"                        # mọi file notebook ghi ra nằm ở đây
CACHE = Path("/tmp/nuse_cache") if ON_KAGGLE else OUT / "cache"
CODE, STATS, RUNS, COMPARE = OUT / "code", OUT / "stats", OUT / "runs", OUT / "comparison"
for path in (OUT, CODE, STATS, RUNS, COMPARE, CACHE):
    path.mkdir(parents=True, exist_ok=True)
os.chdir(OUT)

# Thư mục DIV2K (None = tự tìm trong INPUT_ROOT)
TRAIN_HR_DIR, VAL_HR_DIR = None, None
# Tiếp tục từ phiên trước: thư mục `notebook` trong output cũ (đã Add Input), vd. "/kaggle/input/<ten>/notebook"
RESUME_FROM = "auto"   # "auto": tự tìm trong Input; None: không dùng

# ---- Thí nghiệm ----
METHODS = ["average_pool", "bicubic"]               # cách tạo LR
BASELINES = ["SPANV2", "DISP", "PLKSR-Tiny"]        # chỉ dùng checkpoint cũ / số đã báo cáo (LR average pool)
OLD_RUNS = "reuse"   # "reuse": baseline LR average_pool lấy từ checkpoint cũ nếu tìm thấy; "retrain": train lại tất cả
NO_TRAIN = {m: list(BASELINES) for m in METHODS}     # không train baseline nào trong notebook này
SHOW_OLD_PLUGIN = True                               # thêm dòng "backbone + NuSE (k=8)" của thí nghiệm cũ (LR average_pool)

# ---- Biến thể v3 của NuSE-E. Cùng quy tắc suy kiến trúc từ phổ PCA (mục 4 thiết kế v2); chỉ khác các siêu tham số
# mà quy tắc không quyết định: tổng số block N, độ rộng W, kiểu block, mục tiêu năng lượng, trọng số loss alpha.
# Thứ tự trong dict = thứ tự ưu tiên train (hết giờ thì các biến thể cuối không được train).
VARIANTS = {
    "d8":     dict(total_blocks=8, width=32, block="plain"),                                   # sâu thêm 1 block
    "rep7":   dict(total_blocks=7, width=32, block="rep"),                                     # train 3x3+1x1+I, suy luận = v2
    "w40":    dict(total_blocks=5, width=40, block="plain"),                                   # rộng hơn, nông hơn
    "rep8":   dict(total_blocks=8, width=32, block="rep"),
    "lk7":    dict(total_blocks=7, width=32, block="lk7"),                                     # depthwise 7x7 trong block
    "e80":    dict(total_blocks=7, width=32, block="plain", energy_targets=(0.8, 0.9, 1.0)),   # bỏ ngân sách K=2
    "top":    dict(total_blocks=7, width=32, block="plain", alpha="top"),                      # loss = MSE ảnh ở K lớn nhất
    "w36":    dict(total_blocks=6, width=36, block="plain"),
    "w40rep": dict(total_blocks=5, width=40, block="rep"),
}
PHASE_B_TOP = 2            # xong LR average pool thì train thêm trên LR bicubic cho N biến thể tốt nhất

# ---- Ràng buộc cứng: kiểm trước khi train (tham số, latency) và sau khi train (đơn điệu theo K, khớp LR)
PARAM_CAP = 156856         # tham số ở K lớn nhất <= DISP + NuSE k=8; biến thể nặng hơn không được train
LATENCY_REF = "DISP + NuSE k=8 (cũ)"   # trần latency @LR 256² fp32: đo lại từ checkpoint cũ nếu có, không thì dùng số dưới
LATENCY_CAP_MS = 8.44      # DISP + NuSE k=8 trên T4 (comparison/latency.csv cũ)
CONSISTENCY_TOL = 1e-5     # max|A(SR) - y| cho phép

# ---- Giao thức train (giống các run cũ) ----
EPOCHS, BATCH, REPEAT = 100, 128, 20                 # 800 ảnh × 20 / 128 = 125 iteration mỗi epoch
LR, MIN_LR, GRAD_CLIP, SEED = 1e-3, 1e-6, 1.0, 2026
VAL_EVERY, CHECKPOINT_EVERY = 5, 10                  # đánh giá đủ 100 ảnh (chọn best.pt) mỗi 5 epoch, như run cũ
QUICK_VAL_IMAGES, QUICK_VAL_CROP = 10, 480           # mỗi epoch: PSNR/SSIM trên 10 crop 480×480 để theo dõi tiến độ
AMP = False                                          # True nhanh hơn ~2×; phải giống nhau cho mọi model
WORKERS = min(4, os.cpu_count() or 1)
SESSION_HOURS = 11.5                                 # Kaggle: dừng nhận run mới khi gần hết phiên

# ---- Quy tắc kiến trúc NuSE-E (thiết kế v2, mục 4) ----
DESIGN = dict(energy_targets=(0.5, 0.8, 0.9, 1.0), total_blocks=7, r2_min=0.05, rel_tol=1e-6, min_share=0.5,
              width=32, block="plain", alpha="uniform")   # = bản v2 đã train (tham chiếu)
STATS_REPEAT, STATS_FOLDS = 4, 5

# ---- Đánh giá ----
USE_LPIPS, USE_FID = True, True                      # FID trên patch 256×256 không chồng lấn
BENCHMARK_NAMES = ["Set5", "Set14", "B100", "BSD100", "Urban100", "Manga109"]
SAVE_SR_IMAGES = False

# ---- Latency (giống notebook cũ) ----
LATENCY_SIZES = [(128, 128), (256, 256), (339, 510)]
LATENCY_PRECISIONS = ["fp32", "fp16"]
WARMUP, ITERS, ROUNDS = 30, 300, 10
CPU_LATENCY_SIZE, CPU_ITERS = (128, 128), 20
FLOPS_SIZE = (256, 256)

if SMOKE:                                            # chỉ để kiểm tra code chạy hết các bước
    EPOCHS, BATCH, REPEAT, VAL_EVERY, WORKERS, STATS_REPEAT = 2, 2, 1, 1, 0, 1
    LATENCY_SIZES, WARMUP, ITERS, ROUNDS, CPU_ITERS = [(32, 32)], 1, 2, 1, 2
    VARIANTS, PHASE_B_TOP = {k: VARIANTS[k] for k in ("d8", "rep7", "e80")}, 1
MAX_ITERS = 2 if SMOKE else None
NUSE = ["NuSE-E", *[f"NuSE-E/{v}" for v in VARIANTS]]   # "NuSE-E" = bản v2; "NuSE-E/<tên>" = biến thể v3
MODELS = NUSE + BASELINES

def design_of(model):
    return {**DESIGN, **VARIANTS[model.split("/")[1]]} if "/" in model else dict(DESIGN)

def slug(model):
    return {"SPANV2": "spanv2", "DISP": "disp", "PLKSR-Tiny": "plksr_tiny"}.get(model, "nuse_e" + model[6:].replace("/", "_"))

def arch_path(method, model):
    return STATS / method / ("arch.json" if model == "NuSE-E" else f"arch_{model.split('/')[1]}.json")
print(f"Kaggle={ON_KAGGLE} | output -> {OUT} | cache -> {CACHE}")

In [ ]:
# ============================== Môi trường ==============================
import importlib.util
missing = [pip for mod, pip in [("lpips", "lpips"), ("torchmetrics", "torchmetrics"), ("torch_fidelity", "torch-fidelity")]
           if importlib.util.find_spec(mod) is None]
if missing and not SMOKE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
import torch
print("torch", torch.__version__, "| GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Thư viện

Toàn bộ code (dữ liệu, toán tử, thống kê PCA null space, quy tắc kiến trúc, NuSE-E, baseline, metric, latency, vòng train) nằm trong `code/nuse_lib.py`. Ô dưới ghi file này ra đĩa để các tiến trình train trên từng GPU dùng chung.

In [ ]:
%%writefile code/nuse_lib.py
"""NuSE-E and baselines: data, PCA null-space statistics, models, training and evaluation.

Written by the notebook into <OUT>/code/nuse_lib.py. Design reference:
NuSE-E_thiet_ke_v2.md (theory, sections 2 and 4) and IMPLEMENTATION_SPEC.md.
"""
import copy
import csv
import json
import math
import os
import random
import statistics
import time
from collections import OrderedDict
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

S = 4                                   # scale
IMAGE_EXT = {".png", ".jpg", ".jpeg", ".bmp"}
METHODS = ("average_pool", "bicubic")


# =============================================================== utilities
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed % 2**32)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    def safe(v):
        if isinstance(v, float) and not math.isfinite(v):
            return str(v)
        if isinstance(v, dict):
            return {str(k): safe(x) for k, x in v.items()}
        if isinstance(v, (list, tuple)):
            return [safe(x) for x in v]
        if isinstance(v, Path):
            return str(v)
        if isinstance(v, torch.Tensor):
            return safe(v.tolist())
        return v
    path.write_text(json.dumps(safe(value), indent=2, ensure_ascii=False) + "\n")


def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    keys = list(rows[0])
    for row in rows[1:]:
        keys += [k for k in row if k not in keys]
    with path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)


def image_files(folder):
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMAGE_EXT)


def load_rgb(path):
    with Image.open(path) as image:
        return torch.from_numpy(np.array(image.convert("RGB"), dtype=np.uint8, copy=True)).permute(2, 0, 1)


def to_bmp(args):
    """(source, folder) -> cached BMP path; PNG 2K decoding is slow, BMP reads almost instantly."""
    source, folder = args
    target = Path(folder) / (Path(source).stem + ".bmp")
    if not target.exists():
        with Image.open(source) as image:
            image.convert("RGB").save(target.with_suffix(".partial"), format="BMP")
        target.with_suffix(".partial").rename(target)
    return str(target)


def save_png(path, image):
    """Clamp/round for viewing only; metrics never use this."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pixels = image.detach().cpu().clamp(0, 1).mul(255).round().byte().permute(1, 2, 0).numpy()
    Image.fromarray(pixels).save(path)


# =============================================================== operators
def avg_down(x):
    return F.avg_pool2d(x, S)


def bicubic_down(x):
    """PyTorch antialiased bicubic x1/4 (a = -0.5 inside the image, same as MATLAB imresize)."""
    return F.interpolate(x, scale_factor=1 / S, mode="bicubic", align_corners=False, antialias=True)


def degrade(x, method):
    return avg_down(x) if method == "average_pool" else bicubic_down(x)


class AvgPoolOp:
    name = "average_pool"

    def forward(self, x):
        return avg_down(x)

    def pinv(self, y):
        return y.repeat_interleave(S, -2).repeat_interleave(S, -1)


class BicubicOp:
    """B(x) = M_h x M_w^T (Thm. 5); exact separable pseudo-inverse B+ = M_h^+ (.) M_w^+T.

    M is extracted from bicubic_down itself (impulse responses), so B is exactly the
    function that generates the LR data. Matrices are cached per (H, W, device, dtype).
    """
    name = "bicubic"

    def __init__(self):
        self.cpu_cache = {}            # (H, W) -> FP64 matrices on CPU; one entry per image size

    @staticmethod
    def matrix(n):
        eye = torch.eye(n, dtype=torch.float64).reshape(n, 1, 1, n).expand(n, 1, S, n)
        out = bicubic_down(eye)                                        # (n,1,1,n/S)
        return out.reshape(n, n // S).T.contiguous()                   # (n/S, n)

    def mats(self, h, w, device, dtype):
        if (h, w) not in self.cpu_cache:
            mh, mw = self.matrix(h), self.matrix(w)
            self.cpu_cache[(h, w)] = tuple(t.float() for t in (mh, mw, torch.linalg.pinv(mh), torch.linalg.pinv(mw)))
        key = (h, w, str(device), dtype)
        if getattr(self, "_last_key", None) != key:
            self._last = tuple(t.to(device=device, dtype=dtype) for t in self.cpu_cache[(h, w)])
            self._last_key = key
        return self._last

    def forward(self, x):
        return bicubic_down(x)

    def forward_matrix(self, x):
        mh, mw, _, _ = self.mats(x.shape[-2], x.shape[-1], x.device, x.dtype)
        return mh @ x @ mw.T

    def pinv(self, y):
        _, _, ph, pw = self.mats(y.shape[-2] * S, y.shape[-1] * S, y.device, y.dtype)
        return ph @ y @ pw.T


def make_operator(method):
    return AvgPoolOp() if method == "average_pool" else BicubicOp()


# =============================================================== data
class TrainCrops(Dataset):
    """HR crops (default 256 = LR 64 x 4); LR is generated on the GPU by `degrade`.

    Same sampling as the repo DIV2KDataset: per-item RNG seeded by (seed, epoch, index),
    flips + 90-degree rotations, `repeat` crops per image per epoch.
    """

    def __init__(self, files, crop=256, repeat=20, augment=True, seed=2026):
        self.files, self.crop, self.repeat, self.augment, self.seed = list(files), crop, repeat, augment, seed
        self.epoch = 0

    def __len__(self):
        return len(self.files) * self.repeat

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __getitem__(self, index):
        path = self.files[index % len(self.files)]
        rng = random.Random(self.seed + self.epoch * len(self) + index)
        with Image.open(path) as image:
            image = image.convert("RGB")
            w, h = image.size
            left, top = rng.randrange(w - self.crop + 1), rng.randrange(h - self.crop + 1)
            image = image.crop((left, top, left + self.crop, top + self.crop))
            if self.augment:
                if rng.random() < 0.5:
                    image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                if rng.random() < 0.5:
                    image = image.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
                turns = rng.randrange(4)
                if turns:
                    image = image.transpose({1: Image.Transpose.ROTATE_90, 2: Image.Transpose.ROTATE_180,
                                             3: Image.Transpose.ROTATE_270}[turns])
            hr = torch.from_numpy(np.array(image, dtype=np.uint8, copy=True)).permute(2, 0, 1)
        return hr.float().div_(255), index % len(self.files)


class ValImages(Dataset):
    """Full images as uint8 (to keep RAM low), cropped bottom/right to a multiple of 4."""

    def __init__(self, files):
        self.files = list(files)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        hr = load_rgb(self.files[index])
        h, w = hr.shape[-2] // S * S, hr.shape[-1] // S * S
        return hr[:, :h, :w].contiguous(), self.files[index].stem


def as_float(hr_uint8, device):
    return hr_uint8[None].to(device).float().div_(255)


# ---- RAM-resident training set: one flat uint8 file, crops sliced from a memmap (no decoding per crop)
def _decode_flat(path):
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"), dtype=np.uint8).tobytes(), image.size[1], image.size[0]


def build_flat_cache(files, path, workers=4):
    """Decode every training image once into <path> (raw HWC uint8, back to back) + <path>.json index."""
    from concurrent.futures import ProcessPoolExecutor
    path, index_path = Path(path), Path(str(path) + ".json")
    files = [Path(f) for f in files]
    if index_path.exists() and path.exists():
        index = json.loads(index_path.read_text())
        if [e["name"] for e in index] == [f.stem for f in files]:
            return path
    index, offset = [], 0
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as out, ProcessPoolExecutor(max_workers=max(1, workers)) as pool:
        for f, (raw, h, w) in zip(files, pool.map(_decode_flat, [str(f) for f in files], chunksize=4)):
            out.write(raw)
            index.append({"name": f.stem, "offset": offset, "h": h, "w": w})
            offset += len(raw)
    index_path.write_text(json.dumps(index))
    return path


class FlatImages:
    def __init__(self, path):
        self.index = json.loads(Path(str(path) + ".json").read_text())
        self.data = np.memmap(str(path), dtype=np.uint8, mode="r")

    def __len__(self):
        return len(self.index)

    def size(self, i):
        e = self.index[i]
        return e["h"], e["w"]

    def crop(self, i, top, left, c):
        e = self.index[i]
        view = self.data[e["offset"]:e["offset"] + e["h"] * e["w"] * 3].reshape(e["h"], e["w"], 3)
        return torch.from_numpy(np.ascontiguousarray(view[top:top + c, left:left + c])).permute(2, 0, 1)


class RamCropLoader:
    """Same sampling as TrainCrops (per-item RNG seeded by (seed, epoch, index), flips + rotations,
    `repeat` crops per image per epoch); yields uint8 batches (B,3,crop,crop) straight from RAM."""

    def __init__(self, images, batch, seed, crop=256, repeat=20, augment=True):
        self.images, self.batch, self.seed, self.crop, self.repeat, self.augment = images, batch, seed, crop, repeat, augment
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return math.ceil(len(self.images) * self.repeat / self.batch)

    def item(self, index):
        i = index % len(self.images)
        rng = random.Random(self.seed + self.epoch * len(self.images) * self.repeat + index)
        h, w = self.images.size(i)
        left, top = rng.randrange(w - self.crop + 1), rng.randrange(h - self.crop + 1)
        hr = self.images.crop(i, top, left, self.crop)
        if self.augment:
            if rng.random() < 0.5:
                hr = hr.flip(-1)
            if rng.random() < 0.5:
                hr = hr.flip(-2)
            turns = rng.randrange(4)
            if turns:
                hr = torch.rot90(hr, turns, (-2, -1))
        return hr.contiguous(), i

    def __iter__(self):
        n = len(self.images) * self.repeat
        order = torch.randperm(n, generator=torch.Generator().manual_seed(self.seed + self.epoch)).tolist()
        for start in range(0, n, self.batch):
            items = [self.item(j) for j in order[start:start + self.batch]]
            yield torch.stack([hr for hr, _ in items]), torch.tensor([i for _, i in items])


def train_loader(files, *, batch, workers, seed, crop=256, repeat=20, augment=True):
    data = TrainCrops(files, crop, repeat, augment, seed)
    loader = DataLoader(data, batch_size=batch, shuffle=True, num_workers=workers, pin_memory=True,
                        drop_last=False, persistent_workers=workers > 0,
                        generator=torch.Generator().manual_seed(seed))
    return data, loader


# =============================================================== null-space statistics
def helmert(scale=S, dtype=torch.float64):
    """Q (q, q-1): Q^T Q = I, 1^T Q = 0."""
    q = scale ** 2
    basis = torch.zeros(q, q - 1, dtype=dtype)
    for j in range(1, q):
        norm = math.sqrt(j * (j + 1))
        basis[:j, j - 1] = 1 / norm
        basis[j, j - 1] = -j / norm
    return basis


def to_blocks(x):
    """(B,C,H,W) -> (B,C,16,H/4,W/4), pixel index p = dy*4 + dx."""
    b, c, h, w = x.shape
    return F.pixel_unshuffle(x, S).reshape(b, c, S * S, h // S, w // S)


def d4_permutations():
    idx = torch.arange(S * S).reshape(S, S)
    mats = []
    for turns in range(4):
        for flip in (False, True):
            perm = torch.rot90(idx, turns, (0, 1))
            perm = perm.flip(1) if flip else perm
            p = torch.zeros(S * S, S * S, dtype=torch.float64)
            p[torch.arange(S * S), perm.flatten()] = 1
            mats.append(p)
    return mats


def null_coordinates(hr, method, op, q):
    """Per-channel null-space coordinates u of every block: (C, N, d)."""
    if method == "average_pool":
        blocks = to_blocks(hr)                                         # (B,C,16,h,w)
        u = torch.einsum("bcphw,pd->cbhwd", blocks, q)
    else:
        r = hr - op.pinv(op.forward(hr))
        u = to_blocks(r).permute(1, 0, 3, 4, 2)                        # (C,B,h,w,16)
    return u.reshape(u.shape[0], -1, u.shape[-1])


def lr_windows(lr):
    """Per-channel 3x3 replicate-padded LR windows + 1: (C, N, 10), row-major taps."""
    b, c, h, w = lr.shape
    x = F.pad(lr.reshape(b * c, 1, h, w), (1, 1, 1, 1), mode="replicate")
    win = F.unfold(x, 3).reshape(b, c, 9, h * w).permute(1, 0, 3, 2).reshape(c, b * h * w, 9)
    return torch.cat([win, torch.ones_like(win[..., :1])], -1)


@torch.no_grad()
def fit_statistics(files, method, device, *, repeat=4, folds=5, batch=16, workers=2, seed=2026, log=print):
    """Pass 1: D4-symmetrised PCA of the null-space residual. Pass 2: Wiener weights + k-fold R^2.

    Training HR only (I8). FP64 accumulation. Returns a dict saved as stats.pt.
    """
    op = make_operator(method)
    q = helmert().to(device)
    d = 15 if method == "average_pool" else 16
    frame = q if method == "average_pool" else torch.eye(16, dtype=torch.float64, device=device)
    if isinstance(files, (str, Path)):                                  # flat uint8 cache (build_flat_cache)
        images = FlatImages(files)
        loader = RamCropLoader(images, batch, seed, 256, repeat, augment=False)
        names = [e["name"] for e in images.index]
    else:
        data = TrainCrops(files, 256, repeat, augment=False, seed=seed)
        loader = DataLoader(data, batch_size=batch, shuffle=False, num_workers=workers)
        names = [Path(p).name for p in files]

    def to_f64(hr):
        return hr.to(device, torch.float64) / (255 if hr.dtype == torch.uint8 else 1)
    n, total, outer = 0, None, None
    t0 = time.time()
    for step, (hr, _) in enumerate(loader):
        hr = to_f64(hr)
        u = null_coordinates(hr, method, op, q)                         # (C,N,d)
        if total is None:
            total = torch.zeros(u.shape[0], d, dtype=torch.float64, device=device)
            outer = torch.zeros(u.shape[0], d, d, dtype=torch.float64, device=device)
        total += u.sum(1)
        outer += torch.einsum("cni,cnj->cij", u, u)
        n += u.shape[1]
        if step % 50 == 0:
            log(f"[stats {method}] pass 1 batch {step + 1}/{len(loader)} ({time.time() - t0:.0f}s)")
    mean = total / n
    cov = outer / n - mean[:, :, None] * mean[:, None, :]
    cov = (cov + cov.transpose(1, 2)) / 2
    reps = [(frame.T @ p.to(device) @ frame) for p in d4_permutations()]
    cov_sym = sum(r @ cov @ r.T for r in reps) / len(reps)
    mean_sym = sum(mean @ r.T for r in reps) / len(reps)
    lam, vec = torch.linalg.eigh(cov_sym)
    lam, vec = lam.flip(-1).clamp_min(0), vec.flip(-1)                 # (C,d), (C,d,d) columns = v_i
    invariance = max(((r @ cov_sym @ r.T) - cov_sym).abs().max().item() for r in reps) / cov_sym.abs().max().item()

    xtx = torch.zeros(folds, 3, 10, 10, dtype=torch.float64, device=device)
    xty = torch.zeros(folds, 3, 10, d, dtype=torch.float64, device=device)
    ysum = torch.zeros(folds, 3, d, dtype=torch.float64, device=device)
    ysq = torch.zeros(folds, 3, d, dtype=torch.float64, device=device)
    cnt = torch.zeros(folds, dtype=torch.float64, device=device)
    t0 = time.time()
    for step, (hr, image_index) in enumerate(loader):
        hr = to_f64(hr)
        lr = degrade(hr, method)
        u = null_coordinates(hr, method, op, q)
        c = torch.einsum("cnd,cdk->cnk", u - mean_sym[:, None, :], vec)  # (C,N,d)
        x = lr_windows(lr)                                              # (C,N,10)
        per = (lr.shape[-2] * lr.shape[-1])
        fold_of = (image_index % folds).to(device).repeat_interleave(per)
        for f in range(folds):
            sel = fold_of == f
            if sel.any():
                xs, cs = x[:, sel], c[:, sel]
                xtx[f] += xs.transpose(1, 2) @ xs
                xty[f] += xs.transpose(1, 2) @ cs
                ysum[f] += cs.sum(1)
                ysq[f] += cs.square().sum(1)
                cnt[f] += sel.sum()
        if step % 50 == 0:
            log(f"[stats {method}] pass 2 batch {step + 1}/{len(loader)} ({time.time() - t0:.0f}s)")

    def solve(a, b):
        eps = 1e-8 * torch.diagonal(a, dim1=-2, dim2=-1).sum(-1)[..., None, None] / 10
        return torch.linalg.solve(a + eps * torch.eye(10, dtype=a.dtype, device=a.device), b)

    sse = torch.zeros(3, d, dtype=torch.float64, device=device)
    sst = torch.zeros(3, d, dtype=torch.float64, device=device)
    for f in range(folds):
        keep = [g for g in range(folds) if g != f]
        w = solve(xtx[keep].sum(0), xty[keep].sum(0))                  # (C,10,d)
        sse += ysq[f] - 2 * torch.einsum("cid,cid->cd", w, xty[f]) + torch.einsum("cid,cij,cjd->cd", w, xtx[f], w)
        sst += ysq[f] - ysum[f].square() / cnt[f].clamp_min(1)
    r2 = 1 - sse / sst.clamp_min(1e-30)
    wiener = solve(xtx.sum(0), xty.sum(0))                              # (C,10,d)
    return {"method": method, "scale": S, "d": d, "mean": mean_sym.cpu(), "components": vec.cpu(),
            "eigenvalues": lam.cpu(), "wiener": wiener.cpu(), "r2": r2.cpu(), "blocks_per_channel": n,
            "Q": helmert(), "d4_invariance": invariance, "n_train_images": len(names),
            "files": names, "torch_version": torch.__version__}


# =============================================================== design rule (copied from the reference)
def allocate_blocks(energy, linear_r2, groups, total_blocks):
    if total_blocks < len(groups):
        raise ValueError("Need at least one block per budget")
    start, nl = 0, []
    for size in groups:
        nl.append(sum(energy[i] * (1 - max(0.0, linear_r2[i])) for i in range(start, start + size)))
        start += size
    share = [total_blocks * e / sum(nl) for e in nl]
    blocks = [max(1, math.floor(s)) for s in share]
    while sum(blocks) < total_blocks:
        rem = [s - b for s, b in zip(share, blocks)]
        blocks[rem.index(max(rem))] += 1
    while sum(blocks) > total_blocks:
        over = [b - s if b > 1 else -1e9 for s, b in zip(share, blocks)]
        blocks[over.index(max(over))] -= 1
    return blocks, nl


def admissible_cuts(eigenvalues, rel_tol=1e-6):
    d = len(eigenvalues)
    cuts = [k for k in range(1, d) if eigenvalues[k - 1] - eigenvalues[k] > rel_tol * eigenvalues[k - 1]]
    return cuts + [d]


def choose_budgets(eigenvalues, energy_targets=(0.5, 0.8, 0.9, 1.0), rel_tol=1e-6):
    total = sum(eigenvalues)
    cuts = admissible_cuts(eigenvalues, rel_tol)
    budgets = []
    for target in energy_targets:
        k = next(k for k in cuts if sum(eigenvalues[:k]) / total >= target - 1e-12)
        if k not in budgets:
            budgets.append(k)
    return budgets


def derive_architecture(eigenvalues, linear_r2, *, energy_targets=(0.5, 0.8, 0.9, 1.0), total_blocks=7,
                        r2_min=0.05, rel_tol=1e-6, min_share=0.5):
    budgets = choose_budgets(eigenvalues, energy_targets, rel_tol)

    def nonlinear_energy(lo, hi):
        return sum(eigenvalues[i] * (1 - max(0.0, linear_r2[i])) for i in range(lo, hi))

    while len(budgets) > 1:
        edges = [0] + budgets
        e = [nonlinear_energy(edges[g], edges[g + 1]) for g in range(len(budgets))]
        share = [total_blocks * v / sum(e) for v in e]
        g = min(range(len(budgets) - 1), key=lambda i: share[i])
        if share[g] >= min_share and len(budgets) <= total_blocks:
            break
        budgets.pop(g)
    groups = [b - a for a, b in zip([0] + budgets[:-1], budgets)]
    blocks, nonlinear = allocate_blocks(eigenvalues, linear_r2, groups, total_blocks)
    linear, group_r2, start = [], [], 0
    for size in groups:
        lam = eigenvalues[start:start + size]
        r2 = sum(l * max(0.0, r) for l, r in zip(lam, linear_r2[start:start + size])) / sum(lam)
        group_r2.append(r2)
        linear.append(r2 >= r2_min)
        start += size
    return {"budgets": budgets, "groups": groups, "blocks": blocks, "linear": linear,
            "group_linear_r2": group_r2, "nonlinear_energy": nonlinear,
            "rule": {"energy_targets": list(energy_targets), "total_blocks": total_blocks,
                     "r2_min": r2_min, "rel_tol": rel_tol, "min_share": min_share}}


def architecture_from_stats(stats, width=32, block="plain", alpha="uniform", **rule):
    """Rule output plus the trunk hyper-parameters (width, block kind, loss weights) that the rule does not set."""
    lam = stats["eigenvalues"].mean(0).tolist()
    r2 = stats["r2"].mean(0).tolist()
    arch = derive_architecture(lam, r2, **rule)
    arch["method"] = stats["method"]
    arch["retained_energy"] = [sum(lam[:k]) / sum(lam) for k in arch["budgets"]]
    arch["width"], arch["block"], arch["alpha"] = width, block, alpha
    return arch


def loss_alpha(spec, n_budgets):
    """Thm. 4 holds for any alpha >= 0: 'uniform' = every budget counts once; 'top' = image MSE at K max only."""
    if spec in (None, "uniform"):
        return [1.0] * n_budgets
    if spec == "top":
        return [0.0] * (n_budgets - 1) + [1.0]
    return list(spec)


# =============================================================== NuSE-E
class RepConv(nn.Module):
    """Conv3x3 trained as 3x3 + 1x1 + identity (structural re-parameterisation, RepVGG / ECB style);
    `fuse()` folds the three branches into one Conv3x3, so inference cost equals a plain Conv3x3."""

    def __init__(self, width):
        super().__init__()
        self.conv3 = nn.Conv2d(width, width, 3, padding=1)
        self.conv1 = nn.Conv2d(width, width, 1)

    def forward(self, x):
        return self.conv3(x) + self.conv1(x) + x

    @torch.no_grad()
    def fuse(self):
        width = self.conv3.in_channels
        conv = nn.Conv2d(width, width, 3, padding=1).to(self.conv3.weight.device)
        weight = self.conv3.weight + F.pad(self.conv1.weight, [1, 1, 1, 1])
        weight[torch.arange(width), torch.arange(width), 1, 1] += 1.0
        conv.weight.copy_(weight)
        conv.bias.copy_(self.conv3.bias + self.conv1.bias)
        return conv


class ResBlock(nn.Module):
    """Generic residual block (ResNet / EDSR): Conv3x3 -> ReLU -> Conv3x3, identity skip.

    kind = "plain": as above. "rep": both convs are RepConv while training (same inference block).
    "lk7": Conv3x3 -> ReLU -> depthwise Conv7x7 -> Conv3x3 (larger receptive field for +50 params/channel).
    """

    def __init__(self, width, kind="plain"):
        super().__init__()
        conv = RepConv if kind == "rep" else (lambda w: nn.Conv2d(w, w, 3, padding=1))
        self.conv1 = conv(width)
        self.conv2 = conv(width)
        self.dw = nn.Conv2d(width, width, 7, padding=3, groups=width) if kind == "lk7" else None

    def forward(self, x):
        f = F.relu(self.conv1(x))
        if self.dw is not None:
            f = self.dw(f)
        return x + self.conv2(f)

    def fuse(self):
        for name in ("conv1", "conv2"):
            if isinstance(getattr(self, name), RepConv):
                setattr(self, name, getattr(self, name).fuse())

    @staticmethod
    def inference_params(width, kind):
        return 2 * (9 * width * width + width) + (49 * width + width if kind == "lk7" else 0)


class GroupHead(nn.Module):
    """c_G = sqrt(lambda_G) * (Conv1x1(f_g) [+ L_g(y)]); readout zero-initialised."""

    def __init__(self, width, channels, size, std, linear):
        super().__init__()
        self.channels, self.size = channels, size
        self.readout = nn.Conv2d(width, channels * size, 1)
        nn.init.zeros_(self.readout.weight)
        nn.init.zeros_(self.readout.bias)
        self.linear = (nn.Conv2d(channels, channels * size, 3, padding=1, groups=channels,
                                 padding_mode="replicate") if linear else None)
        self.register_buffer("std", std.reshape(1, channels * size, 1, 1).float())

    @torch.no_grad()
    def set_linear(self, weights):
        """weights (C, size, 10): 9 row-major taps + bias, in coefficient units."""
        std = self.std.reshape(self.channels, self.size)
        w = weights.to(std) / std.unsqueeze(-1)
        self.linear.weight.copy_(w[..., :9].reshape(self.channels * self.size, 1, 3, 3))
        self.linear.bias.copy_(w[..., 9].reshape(-1))

    def forward(self, f, y):
        out = self.readout(f)
        if self.linear is not None:
            out = out + self.linear(y)
        return out * self.std


class FixedDecoder(nn.Module):
    """z = mu + U_K c per channel: one grouped 1x1 conv + PixelShuffle. Buffers only."""

    def __init__(self, mean_block, basis):
        super().__init__()
        self.channels = basis.shape[0]
        self.register_buffer("mean_block", mean_block.float())          # (C,16)
        self.register_buffer("basis", basis.float())                    # (C,16,d)

    def forward(self, coeffs, k):
        c = self.channels
        weight = self.basis[..., :k].reshape(c * S * S, k, 1, 1).to(coeffs.dtype)
        return F.pixel_shuffle(F.conv2d(coeffs, weight, self.mean_block.reshape(-1).to(coeffs.dtype), groups=c), S)

    def targets(self, residual_blocks):
        """c* = U^T (x_b - mu): (B,C,16,h,w) -> (B,C,d,h,w)."""
        return torch.einsum("bcphw,cpk->bckhw", residual_blocks - self.mean_block[None, :, :, None, None],
                            self.basis)


class NuSEE(nn.Module):
    kind = "nuse_e"

    def __init__(self, stats, arch, channels=3):
        super().__init__()
        width, kind = arch.get("width", 32), arch.get("block", "plain")
        self.width, self.kind, self.alpha = width, kind, arch.get("alpha", "uniform")
        self.method = stats["method"]
        self.groups, self.blocks_per_stage = tuple(arch["groups"]), tuple(arch["blocks"])
        self.budgets = list(arch["budgets"])
        self.channels = channels
        self.operator = make_operator(self.method)
        mean, vec, lam = stats["mean"], stats["components"], stats["eigenvalues"]
        if self.method == "average_pool":
            q = stats["Q"]
            basis = torch.einsum("pd,cdk->cpk", q, vec)                   # U = Q V
            mean_block = torch.einsum("pd,cd->cp", q, mean)              # mu = Q m
        else:
            basis, mean_block = vec, mean
        self.decoder = FixedDecoder(mean_block, basis)
        self.stem = nn.Conv2d(channels, width, 3, padding=1)
        self.stages = nn.ModuleList(nn.Sequential(*[ResBlock(width, kind) for _ in range(n)]) for n in arch["blocks"])
        std = lam.clamp_min(1e-12).sqrt()
        self.heads, start = nn.ModuleList(), 0
        for size, lin in zip(arch["groups"], arch["linear"]):
            head = GroupHead(width, channels, size, std[:, start:start + size], lin)
            if lin:
                head.set_linear(stats["wiener"][:, :, start:start + size].permute(0, 2, 1))
            self.heads.append(head)
            start += size

    def coefficients(self, y, k):
        g = self.budgets.index(k)
        f, parts = self.stem(y), []
        for j in range(g + 1):
            f = self.stages[j](f)
            parts.append(self.heads[j](f, y).reshape(y.shape[0], self.channels, self.groups[j], *y.shape[-2:]))
        return torch.cat(parts, 2).flatten(1, 2)

    def detail(self, coeffs, k):
        return self.decoder(coeffs, k)

    def consistency(self, z, y):
        if self.method == "average_pool":
            return z + self.operator.pinv(y)                             # Az = 0 (Thm. 2)
        return z + self.operator.pinv(y - self.operator.forward(z))     # Thm. 6

    def forward(self, y, k=None):
        k = self.budgets[-1] if k is None else k
        return self.consistency(self.detail(self.coefficients(y, k), k), y)

    def fuse(self):
        """Inference form: fold RepConv branches. Idempotent; a no-op for plain / lk7 blocks."""
        for stage in self.stages:
            for block in stage:
                block.fuse()
        return self

    def params_for(self, k):
        """Inference parameters (RepConv counted fused) of the model that serves budget k."""
        g = self.budgets.index(k)
        trunk = sum(ResBlock.inference_params(self.width, self.kind) for s in self.stages[:g + 1] for _ in s)
        mods = [self.stem, *self.heads[:g + 1]]
        return trunk + sum(p.numel() for m in mods for p in m.parameters())

    def residual_blocks(self, hr):
        if self.method == "average_pool":
            return to_blocks(hr)                     # U^T removes the block mean (1^T U = 0)
        return to_blocks(hr - self.operator.pinv(self.operator.forward(hr)))

    def loss(self, lr, hr, alpha=None):
        """Avg pool: weighted coefficient MSE (Thm. 4). Bicubic: sum of image MSE over budgets (Thm. 7)."""
        kmax = self.budgets[-1]
        alpha = loss_alpha(self.alpha if alpha is None else alpha, len(self.budgets))
        chat = self.coefficients(lr, kmax).float()
        with torch.autocast(lr.device.type, enabled=False):
            return self._loss(chat, lr.float(), hr.float(), kmax, alpha)

    def _loss(self, chat, lr, hr, kmax, alpha):
        if self.method == "average_pool":
            with torch.no_grad():
                cstar = self.decoder.targets(self.residual_blocks(hr))
            w = torch.zeros(kmax, device=lr.device)
            for k, a in zip(self.budgets, alpha):
                w[:k] += a
            err = (chat.reshape_as(cstar) - cstar).square()
            return (w[None, None, :, None, None] * err).sum() / hr.numel()
        b = lr.shape[0]
        chat = chat.reshape(b, self.channels, kmax, *lr.shape[-2:])
        total = 0.0
        for k, a in zip(self.budgets, alpha):
            sr = self.consistency(self.detail(chat[:, :, :k].flatten(1, 2), k), lr)
            total = total + a * F.mse_loss(sr, hr)
        return total


# =============================================================== baselines
class SPABV2(nn.Module):
    """SPANV2 block as in the previous experiments: y = (x + f3) * Conv1x1(f3)."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.c1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.c2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.c3 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.gate = nn.Conv2d(out_channels, out_channels, 1)
        self.act = nn.ReLU(inplace=True)
        self.shortcut = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x):
        f3 = self.c3(self.act(self.c2(self.act(self.c1(x)))))
        return (self.shortcut(x) + f3) * self.gate(f3)


class SPANV2(nn.Module):
    """SPANV2 (NTIRE 2026 ESR, XiaomiMM), same code as the previous SPANV2 runs."""

    def __init__(self, channels=3, rank=None, width=32, num_blocks=5):
        super().__init__()
        near = channels * S * S
        self.conv_near = nn.Conv2d(channels, near, 3, padding=1, groups=channels)
        with torch.no_grad():
            self.conv_near.weight.zero_()
            self.conv_near.weight[:, 0, 1, 1] = 1
            self.conv_near.bias.zero_()
        self.blocks = nn.Sequential(*(SPABV2(channels if i == 0 else width, width) for i in range(num_blocks)))
        fused = near + width
        self.fuse_depthwise = nn.Conv2d(fused, fused, 3, padding=1, groups=fused)
        self.fuse_pointwise = nn.Conv2d(fused, channels * S * S if rank is None else channels * rank, 1)
        self.upsample = nn.PixelShuffle(S) if rank is None else nn.Identity()

    def forward(self, lr):
        x = torch.cat((self.conv_near(lr), self.blocks(lr)), dim=1)
        return self.upsample(self.fuse_pointwise(self.fuse_depthwise(x)))


# --- DISP: same layers, parameter names and forward as the official deploy network
# (https://github.com/Amazingren/NTIRE2026_ESR, commit 4592ce6, models/team18_DISP.py,
# MIT License, Copyright (c) 2026 BinRen), written compactly; previous DISP checkpoints load into it.
class DISPConv(nn.Module):
    def __init__(self, c_in, c_out, s=1, bias=True):
        super().__init__()
        self.eval_conv = nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, stride=s, bias=bias)

    def forward(self, x):
        return self.eval_conv(x)


class REECB(nn.Module):
    def __init__(self, in_channels, bias=False, extern_conv=0):
        super().__init__()
        self._extern_conv_block = nn.ModuleList(DISPConv(in_channels, in_channels, bias=bias)
                                                for _ in range(extern_conv))
        self.c1_r = DISPConv(in_channels, in_channels, bias=bias)
        self.c2_r = DISPConv(in_channels, in_channels, bias=bias)
        self.c3_r = DISPConv(in_channels, in_channels, bias=bias)
        self.act1 = nn.SiLU(inplace=True)

    def forward(self, x):
        out2 = self.act1(self.c2_r(self.act1(self.c1_r(x))))
        for extern in self._extern_conv_block:
            out2 = self.act1(extern(out2))
        return self.act1(self.c3_r(out2)) + x


class DISPNet(nn.Module):
    def __init__(self, num_in_ch=3, feature_channels=32, upscale=4, out_channels=None):
        super().__init__()
        out_channels = num_in_ch * upscale ** 2 if out_channels is None else out_channels
        self.conv_1 = DISPConv(num_in_ch, feature_channels, bias=True)
        self.blocks = nn.Sequential(*[REECB(feature_channels, bias=True, extern_conv=1) for _ in range(4)])
        self.conv_2 = nn.Conv2d(feature_channels, feature_channels, 1, 1, 0)
        conv = nn.Conv2d(feature_channels, out_channels, 3, padding=1)
        self.upsampler = (nn.Sequential(conv, nn.PixelShuffle(upscale))
                          if out_channels == num_in_ch * upscale ** 2 else conv)

    def forward(self, x):
        feat = self.conv_1(x)
        out = self.conv_2(self.blocks(feat)) + feat
        return self.upsampler(out)


class DISPSR(nn.Module):
    def __init__(self, channels=3, rank=None, width=32):
        super().__init__()
        self.body = DISPNet(channels, width, S, None if rank is None else channels * rank)

    def forward(self, lr):
        return self.body(lr)


# --- PLKSR-Tiny: layout of the previous PLKSR-Tiny checkpoints (dim 64, 12 blocks, 13x13 partial
# large kernel on 16 channels, no attention), following PLKSR (Lee et al., 2024,
# https://github.com/dslisleedh/PLKSR).
class PLKConv2d(nn.Module):
    def __init__(self, dim, kernel_size):
        super().__init__()
        self.conv = nn.Conv2d(dim, dim, kernel_size, 1, kernel_size // 2)
        nn.init.trunc_normal_(self.conv.weight, std=0.02)
        self.idx = dim

    def forward(self, x):
        x1, x2 = torch.split(x, [self.idx, x.size(1) - self.idx], dim=1)
        return torch.cat([self.conv(x1), x2], dim=1)


class PLKBlock(nn.Module):
    def __init__(self, dim, kernel_size, split_ratio):
        super().__init__()
        self.channe_mixer = nn.Sequential(nn.Conv2d(dim, dim * 2, 3, 1, 1), nn.GELU(),
                                          nn.Conv2d(dim * 2, dim, 3, 1, 1))
        self.lk = PLKConv2d(int(dim * split_ratio), kernel_size)
        self.refine = nn.Conv2d(dim, dim, 1, 1, 0)

    def forward(self, x):
        return x + self.refine(self.lk(self.channe_mixer(x)))


class PLKSRNet(nn.Module):
    def __init__(self, dim=64, n_blocks=12, kernel_size=13, split_ratio=0.25, out_channels=48):
        super().__init__()
        self.out_channels = out_channels
        self.feats = nn.Sequential(nn.Conv2d(3, dim, 3, 1, 1),
                                   *[PLKBlock(dim, kernel_size, split_ratio) for _ in range(n_blocks)],
                                   nn.Conv2d(dim, out_channels, 3, 1, 1))
        nn.init.trunc_normal_(self.feats[0].weight, std=0.02)
        nn.init.trunc_normal_(self.feats[-1].weight, std=0.02)

    def forward(self, x):
        if self.out_channels == 3 * S * S:
            return F.pixel_shuffle(self.feats(x) + x.repeat_interleave(S * S, dim=1), S)
        return self.feats(x)


class PLKSRTiny(nn.Module):
    def __init__(self, channels=3, rank=None, width=64):
        super().__init__()
        self.body = PLKSRNet(dim=width, out_channels=channels * S * S if rank is None else channels * rank)

    def forward(self, lr):
        return self.body(lr)


BACKBONES = {"SPANV2": (SPANV2, 32), "DISP": (DISPSR, 32), "PLKSR-Tiny": (PLKSRTiny, 64)}


class ImageSR(nn.Module):
    """Plain baseline trained with image MSE. State keys `predictor.*` match the previous checkpoints."""
    kind = "image"

    def __init__(self, name):
        super().__init__()
        cls, width = BACKBONES[name]
        self.predictor = cls(3, None, width)

    def forward(self, lr, k=None):
        return self.predictor(lr)

    def loss(self, lr, hr, alpha=None):
        return F.mse_loss(self.predictor(lr), hr)


class LegacyNuSEPlugin(nn.Module):
    """Previous `backbone + NuSE (k=8)` runs: coefficient head + fixed rank-8 decoder + A+y (average pool).

    Only used to load and re-evaluate the previous checkpoints; buffer names match them.
    """
    kind = "legacy_plugin"

    def __init__(self, name, state):
        super().__init__()
        cls, width = BACKBONES[name]
        rank = state["decoder.components"].shape[-1]
        self.predictor = cls(3, rank, width)
        self.decoder = nn.Module()
        for key in ("Q", "mean", "components", "eigenvalues", "synthesis_basis", "block_mean"):
            self.decoder.register_buffer(key, state[f"decoder.{key}"].clone())
        self.rank = rank
        self.operator = AvgPoolOp()

    def forward(self, lr, k=None):
        coeffs = self.predictor(lr)
        b, _, h, w = coeffs.shape
        coeffs = coeffs.reshape(b, 3, self.rank, h, w)
        blocks = torch.einsum("bckhw,cpk->bcphw", coeffs, self.decoder.synthesis_basis.to(coeffs.dtype))
        blocks = blocks + self.decoder.block_mean[None, :, :, None, None].to(coeffs.dtype)
        return self.operator.pinv(lr) + F.pixel_shuffle(blocks.reshape(b, 3 * 16, h, w), S)


def build_model(name, method, stats=None, arch=None):
    if name.startswith("NuSE-E"):                        # "NuSE-E" or "NuSE-E/<variant>"
        return NuSEE(stats, arch)
    return ImageSR(name)


def budgets_of(model):
    return list(model.budgets) if isinstance(model, NuSEE) else [None]


def load_run(path, stats=None, arch=None):
    """Load a checkpoint written by train_run.py or a previous-experiment best.pt."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if "nuse_e" in ck:                                   # this notebook
        info = ck["nuse_e"]
        if info["model"].startswith("NuSE-E"):
            model = NuSEE(ck["stats"], ck["arch"])
            model.load_state_dict(ck["model"])
            model.fuse()
        else:
            model = ImageSR(info["model"])
            model.load_state_dict(ck["model"])
        return model.eval(), ck
    arch_old = ck["architecture"]                        # previous experiments
    names = {"spanv2": "SPANV2", "disp": "DISP", "plksr_tiny": "PLKSR-Tiny"}
    name = names[arch_old["backbone"]]
    if arch_old.get("null_space", False):
        model = LegacyNuSEPlugin(name, ck["model"])
        model.load_state_dict(ck["model"])
    else:
        model = ImageSR(name)
        model.load_state_dict(ck["model"])
    return model.eval(), ck


# =============================================================== metrics
def _crop(a, b, shave):
    if shave:
        a, b = a[..., shave:-shave, shave:-shave], b[..., shave:-shave, shave:-shave]
    return a, b


def rgb_mse(sr, hr, shave=0):
    sr, hr = _crop(sr, hr, shave)
    return (sr.double() - hr.double()).square().flatten(1).mean(1)


def psnr(sr, hr, shave=0):
    return 10 * torch.log10(1.0 / rgb_mse(sr, hr, shave))


def ssim(sr, hr, shave=0):
    """Gaussian 11x11, sigma 1.5, valid window, population covariance (same as the repo)."""
    sr, hr = _crop(sr, hr, shave)
    sr, hr = sr.float(), hr.float()
    size = min(11, *sr.shape[-2:])
    size = size if size % 2 else size - 1
    axis = torch.arange(size, device=sr.device, dtype=sr.dtype) - size // 2
    g = torch.exp(-axis.square() / (2 * 1.5 ** 2))
    g = g / g.sum()
    window = (g[:, None] * g[None, :]).expand(sr.shape[1], 1, size, size)

    def smooth(x):
        return F.conv2d(x, window, groups=sr.shape[1])
    mx, my = smooth(sr), smooth(hr)
    vx = (smooth(sr.square()) - mx.square()).clamp_min(0)
    vy = (smooth(hr.square()) - my.square()).clamp_min(0)
    cov = smooth(sr * hr) - mx * my
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    return (((2 * mx * my + c1) * (2 * cov + c2)) / ((mx.square() + my.square() + c1) * (vx + vy + c2))).flatten(1).mean(1)


def to_y(x):
    """uint8-rounded RGB -> Y (BT.601), range [0,1]."""
    x = x.clamp(0, 1).mul(255).round().div(255)
    return ((65.481 * x[:, 0] + 128.553 * x[:, 1] + 24.966 * x[:, 2] + 16) / 255)[:, None]


class LPIPSMetric(nn.Module):
    """LPIPS v0.1 AlexNet, clamp to [0,1] then [-1,1]."""

    def __init__(self):
        super().__init__()
        import lpips
        with torch.random.fork_rng(devices=[]):
            self.net = lpips.LPIPS(net="alex", version="0.1", verbose=False)
        self.net.requires_grad_(False)
        self.eval()

    @torch.no_grad()
    def forward(self, sr, hr):
        return self.net(sr.float().clamp(0, 1) * 2 - 1, hr.float().clamp(0, 1) * 2 - 1).flatten()


class PatchFID:
    """FID on non-overlapping 256x256 patches (Inception-v3 pool 2048), real features kept across models."""

    def __init__(self, device, patch=256):
        from torchmetrics.image.fid import FrechetInceptionDistance
        self.metric = FrechetInceptionDistance(feature=2048, normalize=True, reset_real_features=False).to(device)
        self.patch, self.has_real = patch, False

    def patches(self, x):
        p = self.patch
        h, w = x.shape[-2] // p * p, x.shape[-1] // p * p
        if h == 0 or w == 0:
            return x.clamp(0, 1)
        x = x[..., :h, :w].clamp(0, 1)
        return x.unfold(2, p, p).unfold(3, p, p).permute(0, 2, 3, 1, 4, 5).reshape(-1, 3, p, p)

    def update(self, image, real):
        for chunk in self.patches(image).split(32):
            self.metric.update(chunk.float(), real=real)

    def compute_and_reset(self):
        value = self.metric.compute().item()
        self.metric.reset()
        return value


# =============================================================== evaluation
@torch.no_grad()
def evaluate(model, images, method, device, *, k=None, lpips=None, fid=None, fid_real=False,
             save_dir=None, amp=False):
    """Per-image metrics on full images. SR is never clamped for PSNR/SSIM (raw float RGB)."""
    op = make_operator(method)
    rows = []
    model.eval()
    for hr_cpu, name in images:
        hr = as_float(hr_cpu, device)
        lr = degrade(hr, method)
        with torch.autocast(device.type, dtype=torch.float16, enabled=amp and device.type == "cuda"):
            sr = model(lr, k) if isinstance(model, NuSEE) else model(lr)
        sr = sr.float()
        back = op.forward(sr)
        row = {"name": name, "rgb_psnr": psnr(sr, hr).item(), "rgb_ssim": ssim(sr, hr).item(),
               "y_psnr_shave4": psnr(to_y(sr), to_y(hr), S).item(), "y_ssim_shave4": ssim(to_y(sr), to_y(hr), S).item(),
               "rgb_psnr_uint8": psnr(sr.clamp(0, 1).mul(255).round().div(255), hr).item(),
               "lr_psnr": psnr(back, lr).item(), "lr_max_abs": (back - lr).abs().max().item(),
               "sr_below0": (sr < 0).float().mean().item(), "sr_above1": (sr > 1).float().mean().item()}
        if lpips is not None:
            row["lpips"] = lpips(sr, hr).item()
        if fid is not None:
            fid.update(sr, real=False)
            if fid_real:
                fid.update(hr, real=True)
        if isinstance(model, NuSEE):
            coeffs = model.coefficients(lr, k)
            cstar = model.decoder.targets(model.residual_blocks(hr))[:, :, :k]
            oracle = model.consistency(model.detail(cstar.flatten(1, 2), k), lr)
            row["oracle_psnr"] = psnr(oracle, hr).item()
            row["truncation_mse"] = rgb_mse(oracle, hr).item()
            row["coefficient_image_mse"] = ((coeffs.reshape_as(cstar) - cstar).double().square().sum() / hr.numel()).item()
        if save_dir is not None:
            save_png(Path(save_dir) / f"{name}.png", sr[0])
        rows.append(row)
    summary = {key: statistics.fmean(r[key] for r in rows) for key in rows[0] if key != "name"}
    summary["lr_max_abs"] = max(r["lr_max_abs"] for r in rows)
    if fid is not None:
        summary["fid_patch256"] = fid.compute_and_reset()
    summary["images"] = len(rows)
    return summary, rows


@torch.no_grad()
def evaluate_bicubic_upsampling(images, method, device, lpips=None, fid=None, fid_real=False):
    class Up(nn.Module):
        def forward(self, lr):
            return F.interpolate(lr, scale_factor=S, mode="bicubic", align_corners=False).clamp(0, 1)
    return evaluate(Up().to(device), images, method, device, lpips=lpips, fid=fid, fid_real=fid_real)


# =============================================================== cost and latency
def count_params(model, k=None):
    if isinstance(model, NuSEE):
        return model.params_for(k)
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def gflops(model, size, device, k=None):
    from torch.utils.flop_counter import FlopCounterMode
    x = torch.rand(1, 3, *size, device=device)
    with FlopCounterMode(display=False) as counter, torch.no_grad():
        model(x, k) if isinstance(model, NuSEE) else model(x)
    return counter.get_total_flops() / 1e9


@torch.no_grad()
def time_cuda(fns, precision, warmup=30, iters=300, rounds=10):
    """CUDA events, batch 1, interleaved over `rounds` so clock drift affects every entry alike."""
    times = {key: [] for key in fns}
    memory = {}
    with torch.autocast("cuda", dtype=torch.float16, enabled=precision == "fp16"):
        for key, fn in fns.items():
            for _ in range(warmup):
                fn()
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            base = torch.cuda.memory_allocated()
            fn()
            torch.cuda.synchronize()
            memory[key] = (torch.cuda.max_memory_allocated() - base) / 2 ** 20
        for _ in range(rounds):
            for key, fn in fns.items():
                events = [(torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True))
                          for _ in range(max(1, iters // rounds))]
                for start, end in events:
                    start.record()
                    fn()
                    end.record()
                torch.cuda.synchronize()
                times[key] += [s.elapsed_time(e) for s, e in events]
    return times, memory


@torch.no_grad()
def time_cpu(fn, iters=20):
    fn()
    out = []
    for _ in range(iters):
        t = time.perf_counter()
        fn()
        out.append((time.perf_counter() - t) * 1e3)
    return out


def stage_functions(label, model, x):
    """Entries to time: total for every model; NuSE-E also split into predictor and decoder + consistency."""
    fns = {}
    for k in budgets_of(model):
        name = label if k is None else f"{label} K={k}"
        if k is None:
            fns[(name, "total")] = (lambda m=model: m(x))
        else:
            coeffs = model.coefficients(x, k)
            fns[(name, "predictor")] = (lambda m=model, k=k: m.coefficients(x, k))
            fns[(name, "decoder + consistency")] = (lambda m=model, k=k, c=coeffs: m.consistency(m.detail(c, k), x))
            fns[(name, "total")] = (lambda m=model, k=k: m(x, k))
    return fns


# =============================================================== training (used by train_run.py)
def quick_images(val, count, crop):
    """First `count` validation images, centre-cropped to crop x crop: fast per-epoch progress check."""
    out = []
    for i in range(min(count, len(val))):
        hr, name = val[i]
        c = min(crop, hr.shape[-2], hr.shape[-1]) // S * S
        top, left = (hr.shape[-2] - c) // 2 // S * S, (hr.shape[-1] - c) // 2 // S * S
        out.append((hr[:, top:top + c, left:left + c].contiguous(), name))
    return out


def progress_text(metrics, model):
    if isinstance(model, NuSEE):
        return " | ".join(f"K{k}: {metrics[f'rgb_psnr_K{k}']:.3f} dB / {metrics[f'rgb_ssim_K{k}']:.4f}" for k in model.budgets)
    return f"{metrics['rgb_psnr']:.3f} dB / {metrics['rgb_ssim']:.4f}"


@torch.no_grad()
def validate(model, images, method, device):
    out = {}
    for k in budgets_of(model):
        summary, _ = evaluate(model, images, method, device, k=k)
        out["rgb_psnr" if k is None else f"rgb_psnr_K{k}"] = summary["rgb_psnr"]
        out["rgb_ssim" if k is None else f"rgb_ssim_K{k}"] = summary["rgb_ssim"]
    if isinstance(model, NuSEE):
        out["rgb_psnr"] = statistics.fmean(out[f"rgb_psnr_K{k}"] for k in model.budgets)
        out["rgb_ssim"] = statistics.fmean(out[f"rgb_ssim_K{k}"] for k in model.budgets)
    return out


def train(cfg, log=print):
    """One run. cfg keys: model, method, run_dir, train_files, val_files, stats, arch, epochs, batch,
    repeat, lr, min_lr, grad_clip, val_every, checkpoint_every, workers, seed, amp, device, max_iters."""
    device = torch.device(cfg["device"])
    run_dir = Path(cfg["run_dir"])
    run_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(cfg["seed"])
    stats = torch.load(cfg["stats"], map_location="cpu", weights_only=False) if cfg.get("stats") else None
    arch = json.loads(Path(cfg["arch"]).read_text()) if cfg.get("arch") else None
    model = build_model(cfg["model"], cfg["method"], stats, arch).to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=cfg["min_lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=cfg["amp"] and device.type == "cuda")
    if cfg.get("train_cache"):                           # flat uint8 cache in RAM: no image decoding per crop
        loader = RamCropLoader(FlatImages(cfg["train_cache"]), cfg["batch"], cfg["seed"], repeat=cfg["repeat"])
    else:
        data, dl = train_loader([Path(p) for p in cfg["train_files"]], batch=cfg["batch"], workers=cfg["workers"],
                                seed=cfg["seed"], repeat=cfg["repeat"])

        class _Wrapped:
            def set_epoch(self, epoch):
                data.set_epoch(epoch)
                dl.generator.manual_seed(cfg["seed"] + epoch)

            def __iter__(self):
                for hr, idx in dl:
                    yield hr.mul(255).round().byte(), idx
        loader = _Wrapped()
    val = ValImages([Path(p) for p in cfg["val_files"]])
    val_images = [val[i] for i in range(len(val))]
    quick = quick_images(val, cfg.get("quick_val_images", 10), cfg.get("quick_val_crop", 480))
    start, best, history = 1, -float("inf"), []
    last = run_dir / "last.pt"
    if last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"])
        scaler.load_state_dict(ck["scaler"])
        start, best, history = ck["epoch"] + 1, ck["best"], ck["history"]
        log(f"resume from epoch {ck['epoch']}")
    meta = {"model": cfg["model"], "method": cfg["method"], "protocol": {k: cfg[k] for k in (
        "epochs", "batch", "repeat", "lr", "min_lr", "grad_clip", "seed", "amp")}}
    t_run = time.time()
    for epoch in range(start, cfg["epochs"] + 1):
        model.train()
        loader.set_epoch(epoch)
        total, count, t0, t_data = 0.0, 0, time.time(), 0.0
        t_step = time.time()
        for step, (hr, _) in enumerate(loader, 1):
            t_data += time.time() - t_step
            hr = hr.to(device, non_blocking=True).float().div_(255)
            with torch.no_grad():
                lr = degrade(hr, cfg["method"])
            with torch.autocast(device.type, dtype=torch.float16, enabled=cfg["amp"] and device.type == "cuda"):
                loss = model.loss(lr, hr)
            loss = loss.float()
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at epoch {epoch} step {step}")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, cfg["grad_clip"])
            scaler.step(opt)
            scaler.update()
            total += loss.item() * hr.shape[0]
            count += hr.shape[0]
            t_step = time.time()
            if cfg.get("max_iters") and step >= cfg["max_iters"]:
                break
        record = {"epoch": epoch, "lr": opt.param_groups[0]["lr"], "train_loss": total / count,
                  "epoch_seconds": time.time() - t0, "data_seconds": t_data}
        sched.step()
        # Progress check every epoch on a few centre crops: no gradient, no RNG use, so training is unchanged.
        record["quick"] = validate(model, quick, cfg["method"], device)
        model.train()
        improved = False
        if epoch % cfg["val_every"] == 0 or epoch == cfg["epochs"]:
            record["validation"] = validate(model, val_images, cfg["method"], device)
            improved = record["validation"]["rgb_psnr"] > best
            best = max(best, record["validation"]["rgb_psnr"])
        history.append(record)
        state = {"nuse_e": meta, "epoch": epoch, "model": model.state_dict(), "optimizer": opt.state_dict(),
                 "scheduler": sched.state_dict(), "scaler": scaler.state_dict(), "best": best,
                 "history": history, "config": cfg, "stats": stats, "arch": arch}
        tmp = run_dir / "last.pt.tmp"
        torch.save(state, tmp)
        os.replace(tmp, last)
        if improved:
            torch.save(state, run_dir / "best.pt")
        if epoch % cfg["checkpoint_every"] == 0:
            torch.save(state, run_dir / f"epoch_{epoch:04d}.pt")
        write_json(run_dir / "history.json", history)
        msg = (f"epoch {epoch}/{cfg['epochs']} loss={record['train_loss']:.6f} "
               f"({record['epoch_seconds']:.0f}s, chờ dữ liệu {t_data:.0f}s) "
               f"| nhanh ({len(quick)} crop): {progress_text(record['quick'], model)}")
        if "validation" in record:
            msg += f" || DIV2K valid đủ {len(val_images)} ảnh: {progress_text(record['validation'], model)}"
        log(msg)
    (run_dir / "DONE").write_text(f"{time.time() - t_run:.0f}s\n")


In [ ]:
%%writefile code/train_run.py
"""Train one run in its own process (one GPU). Usage: python train_run.py <config.json>"""
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import nuse_lib  # noqa: E402

cfg = json.loads(Path(sys.argv[1]).read_text())
log_path = Path(cfg["run_dir"]) / "train.log"
log_path.parent.mkdir(parents=True, exist_ok=True)


def log(message):
    line = f"{time.strftime('%H:%M:%S')} {message}"
    print(line, flush=True)
    with log_path.open("a") as stream:
        stream.write(line + "\n")


log(f"start {cfg['model']} / {cfg['method']} on {cfg['device']}")
nuse_lib.train(cfg, log)
log("done")


In [ ]:
sys.path.insert(0, str(CODE))
import importlib, nuse_lib as L
importlib.reload(L)
import torch

# Tiếp tục từ phiên trước: chép stats/ và runs/ (không ghi đè file đã có)
if RESUME_FROM == "auto":
    found = [p for p in INPUT_ROOT.rglob("runs") if p.is_dir() and (p.parent / "stats").is_dir()
             and not str(p).startswith(str(OUT))] if INPUT_ROOT.is_dir() else []
    RESUME_FROM = found[0].parent if found else None
    print("Phiên trước:", RESUME_FROM or "không có trong Input")
if RESUME_FROM:
    src = Path(RESUME_FROM)
    for sub in ("stats", "runs"):
        if (src / sub).is_dir():
            shutil.copytree(src / sub, OUT / sub, dirs_exist_ok=True)
    print("Đã chép kết quả phiên trước từ", src)

def image_count(folder):
    try:
        return sum(1 for p in Path(folder).iterdir() if p.suffix.lower() in L.IMAGE_EXT)
    except OSError:
        return 0

def find_dir(words, minimum, exclude=("_lr", "lr_", "x2", "x3", "x4", "x8")):
    best = None
    for root, dirs, _ in os.walk(INPUT_ROOT, followlinks=True):
        for name in dirs:
            folder = Path(root) / name
            text = str(folder).lower()
            if any(w in text for w in exclude) or not all(w in text for w in words):
                continue
            n = image_count(folder)
            if n >= minimum and (best is None or n > best[1]):
                best = (folder, n)
    return best[0] if best else None

TRAIN_SRC = Path(TRAIN_HR_DIR) if TRAIN_HR_DIR else find_dir(("div2k", "train", "hr"), 6 if SMOKE else 500)
VAL_SRC = Path(VAL_HR_DIR) if VAL_HR_DIR else find_dir(("div2k", "valid", "hr"), 2 if SMOKE else 50)
assert TRAIN_SRC and VAL_SRC, "Không tìm thấy DIV2K_train_HR / DIV2K_valid_HR trong " + str(INPUT_ROOT)

# PNG 2K giải mã chậm -> chuyển sang BMP một lần (cache ngoài thư mục output)
from concurrent.futures import ProcessPoolExecutor
from PIL import Image

def cache_bmp(src, name):
    folder = CACHE / name
    folder.mkdir(parents=True, exist_ok=True)
    jobs = [(str(p), str(folder)) for p in L.image_files(src)]
    with ProcessPoolExecutor(max_workers=max(1, os.cpu_count() or 1)) as pool:
        return [Path(p) for p in pool.map(L.to_bmp, jobs, chunksize=8)]

# Train: giải mã một lần vào một file uint8 phẳng (6.6 GB với DIV2K); crop lấy thẳng từ RAM, không giải mã lại.
# Phiên trước mỗi crop giải mã lại cả ảnh 2K nên 150 s/epoch; GPU chỉ cần khoảng 20-30 s/epoch.
t0 = time.time()
TRAIN_FILES = L.image_files(TRAIN_SRC)
TRAIN_CACHE = L.build_flat_cache(TRAIN_FILES, CACHE / "div2k_train_u8.bin", workers=max(1, os.cpu_count() or 1))
VAL_FILES = cache_bmp(VAL_SRC, "div2k_valid_bmp")
print(f"Train HR: {TRAIN_SRC} ({len(TRAIN_FILES)} ảnh, cache {TRAIN_CACHE.stat().st_size / 1e9:.1f} GB, {time.time() - t0:.0f}s) "
      f"| Valid HR: {VAL_SRC} ({len(VAL_FILES)} ảnh)")

BENCHMARKS = {}
for name in BENCHMARK_NAMES:
    folder = find_dir((name.lower(),), 5, exclude=("_lr", "lr_", "lrbi", "x2", "x3", "x4", "x8", "bicubic"))
    if folder:
        BENCHMARKS[name] = L.image_files(folder)
print("Benchmark tìm thấy:", {k: len(v) for k, v in BENCHMARKS.items()} or "không có")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Checkpoint của các thí nghiệm cũ (không bắt buộc)

Tìm `best.pt` của `spanv2_x4`, `disp_x4`, `plksr_tiny_x4` và các bản `+NuSE k=8` trong Input. Kiểm tra cấu hình train của chúng có đúng giao thức chung không.

In [ ]:
OLD_NAMES = {"SPANV2": "spanv2_x4", "DISP": "disp_x4", "PLKSR-Tiny": "plksr_tiny_x4",
             "SPANV2 + NuSE k=8 (cũ)": "spanv2_nuse_x4_k8", "DISP + NuSE k=8 (cũ)": "disp_nuse_x4_k8",
             "PLKSR-Tiny + NuSE k=8 (cũ)": "plksr_tiny_nuse_x4_k8"}
OLD = {}
for label, folder in OLD_NAMES.items():
    hits = [p for p in INPUT_ROOT.rglob(f"{folder}/best.pt")] if INPUT_ROOT.is_dir() else []
    if hits:
        OLD[label] = hits[0]
PROTOCOL = {"epochs": EPOCHS, "batch_size": BATCH, "repeat": REPEAT, "lr": LR, "min_lr": MIN_LR, "grad_clip": GRAD_CLIP}
for label, path in OLD.items():
    cfg = torch.load(path, map_location="cpu", weights_only=False)["config"]
    method = ((cfg.get("train_dataset_options") or {}).get("resize") or {}).get("method", "average_pool")
    diff = {k: (cfg.get(k), v) for k, v in PROTOCOL.items() if cfg.get(k) != v}
    print(f"{label:28s} {path}\n   LR = {method}; khác giao thức: {diff or 'không'}")
if not OLD:
    print("Không có checkpoint cũ: mọi baseline sẽ được train trong notebook này.")

## Bước 1: thống kê PCA null space và suy ra kiến trúc NuSE-E

Chỉ dùng HR train (I8). Gồm PCA làm đối xứng D4 trên phần dư null space (average pool: tọa độ Helmert 15 chiều; bicubic: phần dư $x - B^+Bx$ 16 chiều), Wiener và R² kiểm định chéo 5 fold. Kiến trúc lấy từ `derive_architecture`, không chọn tay (I4).

In [ ]:
ARCH = {}
for method in METHODS:
    folder = STATS / method
    stats_path = folder / "stats.pt"
    if not stats_path.exists():
        t0 = time.time()
        stats = L.fit_statistics(TRAIN_CACHE, method, DEVICE, repeat=STATS_REPEAT, folds=STATS_FOLDS,
                                 workers=WORKERS, seed=SEED)
        folder.mkdir(parents=True, exist_ok=True)
        torch.save(stats, stats_path)
        print(f"[{method}] thống kê xong sau {time.time() - t0:.0f}s")
    stats = torch.load(stats_path, map_location="cpu", weights_only=False)
    lam = stats["eigenvalues"].mean(0)
    print(f"\n=== {method}: d = {stats['d']}, {stats['blocks_per_channel']:,} block/kênh, "
          f"sai lệch bất biến D4 = {stats['d4_invariance']:.1e}")
    print("tỉ số λ_i/λ_(i+1):", [round(x, 3) for x in (lam[:-1] / lam[1:]).tolist()])
    print("R² tuyến tính (trung bình kênh):", [round(x, 3) for x in stats["r2"].mean(0).tolist()])
    ARCH[method] = {}
    for name in NUSE:
        arch = L.architecture_from_stats(stats, **design_of(name))
        L.write_json(arch_path(method, name), arch)
        ARCH[method][name] = arch
        print(f"{name:14s} W={arch['width']} {arch['block']:5s} N={sum(arch['blocks'])} alpha={arch['alpha']:7s} | "
              f"K = {arch['budgets']} | block = {arch['blocks']} | L_g = {arch['linear']} | "
              f"năng lượng = {[round(e, 3) for e in arch['retained_energy']]}")

## Bước 2: test chấp nhận

Phải pass hết trước khi train. Mã test theo `IMPLEMENTATION_SPEC.md` mục 7.

In [ ]:
def check(name, ok, value=""):
    print(f"{'PASS' if ok else 'FAIL'}  {name}  {value}")
    return ok

results = []
q = L.helmert()
results.append(check("T2 QᵀQ = I, 1ᵀQ = 0", (q.T @ q - torch.eye(15, dtype=q.dtype)).abs().max() < 1e-12
                     and q.sum(0).abs().max() < 1e-12))
for method in METHODS:
    stats = torch.load(STATS / method / "stats.pt", map_location="cpu", weights_only=False)
    op = L.make_operator(method)
    hr = L.as_float(L.ValImages(VAL_FILES)[0][0], "cpu").double()
    lr_data = L.degrade(hr, method)
    if method == "bicubic":
        results.append(check(f"T1 [{method}] B (ma trận) = hàm tạo LR", (op.forward_matrix(hr) - lr_data).abs().max() < 1e-6,
                             f"{(op.forward_matrix(hr) - lr_data).abs().max():.1e}"))
    results.append(check(f"T3 [{method}] bất biến D4", stats["d4_invariance"] < 1e-12, f"{stats['d4_invariance']:.1e}"))
    y = L.degrade(torch.rand(1, 3, 128, 160), method)
    for name in NUSE:
        arch = ARCH[method][name]
        tag = f"[{method} {name}]"
        model = L.NuSEE(stats, arch).eval()
        u = model.decoder.basis.double()
        results.append(check(f"T4 {tag} UᵀU = I", (torch.einsum("cpk,cpj->ckj", u, u) - torch.eye(u.shape[-1])).abs().max() < 1e-5))
        worst = 0.0
        for seed in range(3):
            torch.manual_seed(seed)
            net = L.NuSEE(stats, arch).eval()
            with torch.no_grad():
                for p in net.parameters():
                    p.normal_(0, 1)
                for k in net.budgets:
                    z = net.detail(net.coefficients(y, k), k)
                    worst = max(worst, ((op.forward(net(y, k)) - y).abs().max() / z.abs().max().clamp_min(1)).item())
        results.append(check(f"T5 {tag} A(SR)=y, trọng số ngẫu nhiên, sai số / max|z|", worst < 1e-6, f"{worst:.1e}"))
        with torch.no_grad():
            k1 = model.budgets[min(1, len(model.budgets) - 1)]
            before = model(y, k1).clone()
            g1 = model.budgets.index(k1)
            for mod in list(model.stages[g1 + 1:]) + list(model.heads[g1 + 1:]):
                for p in mod.parameters():
                    p.add_(torch.randn_like(p))
            results.append(check(f"T7 {tag} output ở K={k1} không phụ thuộc stage sau", torch.equal(before, model(y, k1))))
        model = L.NuSEE(stats, arch)
        W, blk = arch["width"], L.ResBlock.inference_params(arch["width"], arch["block"])
        formula = [28 * W + blk * sum(arch["blocks"][:g + 1]) + sum((3 * W + 3) * n + 30 * n * l for n, l in
                   zip(arch["groups"][:g + 1], arch["linear"][:g + 1])) for g in range(len(arch["budgets"]))]
        params = [model.params_for(k) for k in model.budgets]
        results.append(check(f"T8 {tag} số tham số suy luận theo công thức", params == formula, params))
        results.append(check(f"T9 {tag} mỗi ngân sách thêm ≥ 1 block",
                             min([params[0]] + [b - a for a, b in zip(params, params[1:])]) >= blk))
        if arch["block"] == "rep":
            torch.manual_seed(0)
            net = L.NuSEE(stats, arch).eval()
            with torch.no_grad():
                for p in net.parameters():
                    p.normal_(0, 0.3)
                outs = [net(y, k).clone() for k in net.budgets]
                net.fuse()
                fused = [net(y, k) for k in net.budgets]
                n_params = sum(p.numel() for p in net.parameters())
            err = max(((a - b).abs().max() / a.abs().max()).item() for a, b in zip(outs, fused))
            results.append(check(f"T11 {tag} gộp RepConv: output bằng nhau, tham số = công thức",
                                 err < 1e-5 and n_params == params[-1], f"{err:.1e}, {n_params}"))
        if arch["linear"][0]:
            lr = L.degrade(torch.rand(2, 3, 64, 64, dtype=torch.float64), method)
            g0 = arch["groups"][0]
            with torch.no_grad():
                got = model.eval().coefficients(lr.float(), model.budgets[0]).double().reshape(2, 3, g0, 16, 16)
            x = L.lr_windows(lr)
            want = torch.einsum("cni,cik->cnk", x, stats["wiener"][:, :, :g0]).reshape(3, 2, 16, 16, g0).permute(1, 0, 4, 2, 3)
            rel = ((got - want).abs().max() / want.abs().max()).item()
            results.append(check(f"T10 {tag} lúc khởi tạo head = Wiener", rel < 1e-5, f"{rel:.1e}"))
assert all(results), "Có test FAIL: dừng, không train."

## Bước 2b: sàng ràng buộc cứng trước khi train

Tham số và latency không phụ thuộc trọng số, nên kiểm được trước khi train. Biến thể nào ở K lớn nhất nặng hơn `PARAM_CAP` hoặc chậm hơn trần latency (DISP + NuSE k=8, cùng GPU) thì **không train**. Latency sàng đo nhanh (ít vòng) trên LR 256², fp32; bước 6 đo lại đầy đủ.

In [ ]:
@torch.no_grad()
def quick_ms(fn, warmup=20, iters=100):
    t, _ = L.time_cuda({"x": fn}, "fp32", warmup, iters, 2)
    return statistics.median(t["x"])

cap_ms, cap_src = LATENCY_CAP_MS, "số cũ (T4)"
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    x256 = torch.rand(1, 3, 256, 256, device=DEVICE)
    if LATENCY_REF in OLD:
        ref_model = L.load_run(OLD[LATENCY_REF])[0].to(DEVICE).eval()
        cap_ms, cap_src = quick_ms(lambda: ref_model(x256)), f"đo lại trên {torch.cuda.get_device_name(0)}"
        del ref_model
print(f"Trần: tham số <= {PARAM_CAP:,} | latency @256² fp32 <= {cap_ms:.2f} ms ({cap_src})")
ADMISSIBLE, SCREEN = {}, []
screen_method = "average_pool" if "average_pool" in METHODS else METHODS[0]   # trunk giống nhau giữa hai loại LR
stats_s = torch.load(STATS / screen_method / "stats.pt", map_location="cpu", weights_only=False)
for name in NUSE:
    arch = ARCH[screen_method][name]
    model = L.NuSEE(stats_s, arch).fuse().to(DEVICE).eval()
    kmax = model.budgets[-1]
    params = model.params_for(kmax)
    ms = quick_ms(lambda: model(x256, kmax)) if DEVICE.type == "cuda" else None
    ok = params <= PARAM_CAP and (ms is None or ms <= cap_ms)
    SCREEN.append({"model": name, "params_kmax": params, "ms_256_fp32": ms, "admissible": ok})
    print(f"{'OK  ' if ok else 'LOẠI'} {name:14s} W={arch['width']} {arch['block']:5s} N={sum(arch['blocks'])} K={arch['budgets']} "
          f"| tham số K max {params:,} | {('%.2f ms' % ms) if ms else '—'}")
    del model
ADMISSIBLE = {m: [r["model"] for r in SCREEN if r["admissible"]] for m in METHODS}
L.write_csv(COMPARE / "screen.csv", SCREEN)
assert "NuSE-E" in ADMISSIBLE[screen_method], "Bản tham chiếu v2 phải qua sàng; nếu không thì trần đo sai."

## Bước 3: train

Mỗi run là một tiến trình riêng trên một GPU (2 GPU thì 2 run song song). Ô này in tiến độ mỗi phút. Chạy lại ô sẽ bỏ qua run đã xong và tiếp tục run dở.

**Theo dõi tiến độ:**
- **Mỗi epoch:** log in loss và PSNR / SSIM trên 10 crop 480×480 của DIV2K valid. Với NuSE-E, số được in ở từng K. Số này chỉ để xem model có đang tiến bộ không, không dùng để so với run cũ.
- **Mỗi 5 epoch:** log in thêm PSNR / SSIM trên đủ 100 ảnh DIV2K valid. `best.pt` được chọn theo số này, như các run cũ.

Trên Kaggle, khi chạy bằng Save & Run All, mở trang version và chọn tab **Logs** để xem log trong lúc chạy. Log đầy đủ cũng nằm trong `runs/<lr>/<model>/train.log`.

In [ ]:
PRIORITY = {m: i for i, m in enumerate(MODELS)}       # biến thể theo thứ tự khai báo; baseline (nếu train) sau cùng

def run_dir(method, model):
    return RUNS / method / slug(model)

def uses_old(method, model):
    return OLD_RUNS == "reuse" and method == "average_pool" and model in OLD

def is_done(method, model):
    return (run_dir(method, model) / "DONE").exists()

def best_psnr_kmax(method, model):
    ck = torch.load(run_dir(method, model) / "best.pt", map_location="cpu", weights_only=False)
    kmax = ck["arch"]["budgets"][-1]
    rec = next(r for r in ck["history"] if r["epoch"] == ck["epoch"])
    return rec["validation"][f"rgb_psnr_K{kmax}"]

def job_config(method, model):
    cfg = dict(model=model, method=method, run_dir=str(run_dir(method, model)),
               train_files=[str(p) for p in TRAIN_FILES], val_files=[str(p) for p in VAL_FILES],
               train_cache=str(TRAIN_CACHE),
               stats=str(STATS / method / "stats.pt") if model.startswith("NuSE-E") else None,
               arch=str(arch_path(method, model)) if model.startswith("NuSE-E") else None,
               epochs=EPOCHS, batch=BATCH, repeat=REPEAT, lr=LR, min_lr=MIN_LR, grad_clip=GRAD_CLIP,
               val_every=VAL_EVERY, checkpoint_every=CHECKPOINT_EVERY, workers=WORKERS, seed=SEED, amp=AMP,
               quick_val_images=QUICK_VAL_IMAGES, quick_val_crop=QUICK_VAL_CROP,
               device="cuda" if torch.cuda.is_available() else "cpu", max_iters=MAX_ITERS)
    path = run_dir(method, model) / "config.json"
    L.write_json(path, cfg)
    return path

def wanted(method, model):
    if model.startswith("NuSE-E"):
        if model not in ADMISSIBLE[method]:
            return False
        # Pha A: mọi biến thể trên LR average pool. Pha B (bicubic): bản v2 + PHASE_B_TOP biến thể tốt nhất của pha A.
        return method == "average_pool" or model == "NuSE-E" or "average_pool" not in METHODS
    return model not in NO_TRAIN.get(method, []) and not uses_old(method, model)

jobs = sorted([(m, mdl) for m in METHODS for mdl in MODELS if wanted(m, mdl) and not is_done(m, mdl)],
              key=lambda j: (PRIORITY[j[1]], j[0] != "average_pool"))
phase_b_pending = "bicubic" in METHODS and "average_pool" in METHODS and PHASE_B_TOP > 0
for m in METHODS:
    for mdl in NO_TRAIN.get(m, []):
        if mdl in MODELS and not uses_old(m, mdl):
            note = ("dùng số đã báo cáo [số cũ] ở bước 5" if m == "average_pool"
                    else "không có trong so sánh LR bicubic (chưa từng train)")
            print(f"{mdl} / {m}: không train (NO_TRAIN), không có best.pt -> {note}")
print("Đã xong từ trước (DONE):", [f"{mdl} / {m}" for m in METHODS for mdl in MODELS if is_done(m, mdl)] or "không có")
print("Cần train:", jobs or "không còn run nào")
slots = [str(i) for i in range(torch.cuda.device_count())] or ["cpu"]
start_time, active, shown = time.time(), {}, {}
while jobs or active or phase_b_pending:
    if not jobs and not active and phase_b_pending:
        phase_b_pending = False
        done_a = [n for n in NUSE if n != "NuSE-E" and n in ADMISSIBLE["average_pool"] and is_done("average_pool", n)]
        ranked = sorted(done_a, key=lambda n: -best_psnr_kmax("average_pool", n))
        jobs = [("bicubic", n) for n in ranked[:PHASE_B_TOP] if not is_done("bicubic", n)]
        print("Pha B (LR bicubic) cho biến thể tốt nhất pha A:", jobs or "không có")
    for slot in [s for s in slots if s not in active]:
        if not jobs:
            break
        if (time.time() - start_time) / 3600 > SESSION_HOURS - 1:
            print("Gần hết phiên: không nhận run mới. Chạy tiếp ở phiên sau với RESUME_FROM.")
            jobs, phase_b_pending = [], False
            break
        method, model = jobs.pop(0)
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=slot) if slot != "cpu" else dict(os.environ)
        config_path = job_config(method, model)
        log = open(run_dir(method, model) / "stdout.log", "a")
        active[slot] = (method, model, subprocess.Popen(
            [sys.executable, "-u", str(CODE / "train_run.py"), str(config_path)],
            env=env, stdout=log, stderr=subprocess.STDOUT), log)
        print(f"[GPU {slot}] bắt đầu {model} / {method}")
    time.sleep(1 if SMOKE else 60)
    for slot, (method, model, proc, log) in list(active.items()):
        tail = (run_dir(method, model) / "train.log")
        last = tail.read_text().strip().splitlines()[-1] if tail.exists() and tail.read_text().strip() else ""
        if last and shown.get(slot) != last:
            print(f"[GPU {slot}] {model} / {method}: {last}")
            shown[slot] = last
        if proc.poll() is not None:
            log.close()
            del active[slot]
            if proc.returncode != 0:
                print((run_dir(method, model) / "stdout.log").read_text()[-3000:])
                raise RuntimeError(f"{model} / {method} lỗi (mã {proc.returncode})")
print(f"Train xong sau {(time.time() - start_time) / 3600:.2f} giờ")

## Bước 4: đường học (PSNR validation theo epoch)

In [ ]:
import matplotlib.pyplot as plt
FIG = COMPARE / "figures"
FIG.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, len(METHODS), figsize=(7 * len(METHODS), 4.2), squeeze=False)
for ax, method in zip(axes[0], METHODS):
    for model in MODELS:
        path = run_dir(method, model) / "history.json"
        if not path.exists():
            continue
        hist = [r for r in json.loads(path.read_text()) if "validation" in r]
        if model == "NuSE-E":
            for k in ARCH[method][model]["budgets"]:
                ax.plot([r["epoch"] for r in hist], [r["validation"][f"rgb_psnr_K{k}"] for r in hist],
                        label=f"NuSE-E K={k}", lw=1.6)
        elif model.startswith("NuSE-E"):                     # biến thể: chỉ vẽ K lớn nhất cho đỡ rối
            k = ARCH[method][model]["budgets"][-1]
            ax.plot([r["epoch"] for r in hist], [r["validation"][f"rgb_psnr_K{k}"] for r in hist],
                    ":", label=f"{model} K={k}", lw=1.2)
        else:
            ax.plot([r["epoch"] for r in hist], [r["validation"]["rgb_psnr"] for r in hist], "--", label=model, lw=1.4)
    ax.set(title=f"LR {method}", xlabel="epoch", ylabel="PSNR validation (dB)")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(FIG / "1_training_curves.png", dpi=150); plt.show()

## Bước 4b: chọn biến thể

Trong các biến thể đã qua sàng và train xong, chọn bản có PSNR DIV2K valid ở K lớn nhất cao nhất (lấy từ `best.pt`); bằng nhau thì lấy bản ít tham số hơn. Bản v2 luôn được giữ để đối chiếu.

In [ ]:
SELECTED, SEL_ROWS = {}, []
for method in METHODS:
    cands = []
    for name in NUSE:
        if name in ADMISSIBLE[method] and is_done(method, name):
            ck = torch.load(run_dir(method, name) / "best.pt", map_location="cpu", weights_only=False)
            rec = next(r for r in ck["history"] if r["epoch"] == ck["epoch"])
            arch = ck["arch"]
            row = {"lr": method, "model": name, "width": arch["width"], "block": arch["block"], "blocks": sum(arch["blocks"]),
                   "alpha": arch["alpha"], "budgets": str(arch["budgets"]), "best_epoch": ck["epoch"],
                   "params_kmax": next(r["params_kmax"] for r in SCREEN if r["model"] == name),
                   "psnr_mean_K": rec["validation"]["rgb_psnr"],
                   **{f"psnr_K{k}": rec["validation"][f"rgb_psnr_K{k}"] for k in arch["budgets"]}}
            SEL_ROWS.append(row)
            cands.append((rec["validation"][f"rgb_psnr_K{arch['budgets'][-1]}"], -row["params_kmax"], name))
    SELECTED[method] = max(cands)[2] if cands else None
    print(f"[{method}] chọn: {SELECTED[method]}")
    for c in sorted(cands, reverse=True):
        print(f"   {c[2]:14s} PSNR K max {c[0]:.4f}  tham số {-c[1]:,}")
L.write_csv(COMPARE / "variants.csv", SEL_ROWS)
FULL = {m: {*BASELINES, "NuSE-E", SELECTED[m]} - {None} for m in METHODS}   # được đánh giá đầy đủ (LPIPS, FID, latency mọi K)

## Bước 5: nạp model, đánh giá chất lượng

DIV2K valid 100 ảnh toàn khung (cộng benchmark nếu có).

- PSNR/SSIM RGB trên SR thô, không clamp, shave 0: giống các notebook cũ.
- PSNR/SSIM kênh Y (uint8, shave 4): để so với bài báo.
- LPIPS (AlexNet).
- FID trên patch 256×256.
- Độ khớp LR: `max|A(SR) − y|`.

NuSE-E được đánh giá ở mọi K. Các biến thể không được chọn chỉ đo PSNR/SSIM (không LPIPS, FID) cho nhanh.

In [ ]:
MODELS_EVAL = {}            # method -> {label: (model, k)}
NAME_OF = lambda label: label.split(" K=")[0]
for method in METHODS:
    entries = {}
    for model_name in MODELS:
        path = OLD[model_name] if uses_old(method, model_name) else run_dir(method, model_name) / "best.pt"
        if not Path(path).exists() or (model_name.startswith("NuSE-E") and not is_done(method, model_name)):
            print(f"không có {model_name} / {method} -> bỏ qua trong so sánh LR {method}")
            continue
        model, _ = L.load_run(path)
        model = model.to(DEVICE).eval()
        for k in L.budgets_of(model):
            entries[model_name if k is None else f"{model_name} K={k}"] = (model, k)
    if method == "average_pool" and SHOW_OLD_PLUGIN:
        for label, path in OLD.items():
            if "NuSE k=8" in label:
                entries[label] = (L.load_run(path)[0].to(DEVICE).eval(), None)
    MODELS_EVAL[method] = entries
    print(method, "->", list(entries))

lpips_metric = L.LPIPSMetric().to(DEVICE) if USE_LPIPS else None
val_images = [img for img in L.ValImages(VAL_FILES)]          # uint8 trong RAM
REPORT, PER_IMAGE = {}, {}
for method in METHODS:
    REPORT[method], PER_IMAGE[method] = {}, {}
    sets = {"DIV2K-valid": val_images, **{n: [img for img in L.ValImages(f)] for n, f in BENCHMARKS.items()}}
    for set_name, images in sets.items():
        # FID cần đủ patch: chỉ tính trên bộ có >= 50 ảnh (DIV2K valid). Đặc trưng HR tính một lần cho mỗi bộ.
        fid = L.PatchFID(DEVICE) if USE_FID and len(images) >= 50 else None
        summary, rows = L.evaluate_bicubic_upsampling(images, method, DEVICE, lpips=lpips_metric, fid=fid, fid_real=True)
        REPORT[method][(set_name, "Bicubic")] = summary
        for label, (model, k) in MODELS_EVAL[method].items():
            t0 = time.time()
            full = NAME_OF(label) in FULL[method] or "(cũ)" in label
            save = OUT / "sr_images" / method / label.replace(" ", "_") if SAVE_SR_IMAGES and set_name == "DIV2K-valid" else None
            summary, rows = L.evaluate(model, images, method, DEVICE, k=k, lpips=lpips_metric if full else None,
                                       fid=fid if full else None, save_dir=save)
            REPORT[method][(set_name, label)] = summary
            PER_IMAGE[method][(set_name, label)] = rows
            print(f"[{method}] {set_name:12s} {label:30s} PSNR {summary['rgb_psnr']:.4f}  SSIM {summary['rgb_ssim']:.4f}  "
                  f"LPIPS {summary.get('lpips', float('nan')):.4f}  FID {summary.get('fid_patch256', float('nan')):.2f}  "
                  f"({time.time() - t0:.0f}s)")
    rows = [{"set": s, "model": lab, **v} for (s, lab), v in REPORT[method].items()]
    L.write_csv(COMPARE / method / "quality.csv", rows)
    L.write_csv(COMPARE / method / "per_image.csv",
                [{"set": s, "model": lab, **r} for (s, lab), rr in PER_IMAGE[method].items() for r in rr])

### Baseline khi không có file `best.pt`

Nếu không Add Input checkpoint cũ, bảng LR average pool dùng **số đã báo cáo** của các run cũ (lấy từ `comparison/summary.json` và `latency.csv`, chép sẵn vào ô dưới), gắn nhãn `[số cũ]`.

Những số này so được với NuSE-E vì cùng DIV2K valid, cùng định nghĩa metric (PSNR/SSIM RGB thô, shave 0, LPIPS AlexNet), cùng số tham số và cách đếm FLOPs. Chúng **không có** Y-PSNR, FID, so sánh từng ảnh và hình trực quan.

Latency cũ đo trên GPU khác (T4 cho DISP, RTX 3090 cho PLKSR-Tiny và SPANV2), nên chỉ in ra để tham khảo, không đưa vào bảng chung và hình latency.

Muốn so sánh đầy đủ trên cùng GPU thì Add Input `nuse_old_runs_kaggle.zip`, gói `best.pt` của 6 run cũ (57 MB).

In [ ]:
# Số đã báo cáo của các run cũ: comparison/summary.json + latency.csv (LR average_pool, DIV2K valid 100 ảnh)
OLD_REPORTED = {
    'DISP': {'rgb_psnr': 27.520466, 'rgb_ssim': 0.782759, 'lpips': 0.344047, 'lr_max_abs': 0.10915189981460571, 'params': 163792, 'gflops': 21.3868, 'gpu': 'Tesla T4', 'ms_fp32_339x510': 24.042, 'ms_fp16_339x510': 14.673},
    'DISP + NuSE': {'rgb_psnr': 28.111805, 'rgb_ssim': 0.799086, 'lpips': 0.284612, 'lr_max_abs': 2.384185791015625e-07, 'params': 156856, 'gflops': 20.5311, 'gpu': 'Tesla T4', 'ms_fp32_339x510': 24.651, 'ms_fp16_339x510': 15.698},
    'PLKSR-Tiny': {'rgb_psnr': 28.486712, 'rgb_ssim': 0.809117, 'lpips': 0.272896, 'lr_max_abs': 0.0018892884254455566, 'params': 2370544, 'gflops': 310.2694, 'gpu': 'NVIDIA GeForce RTX 3090', 'ms_fp32_339x510': 76.988, 'ms_fp16_339x510': 51.477},
    'PLKSR-Tiny + NuSE': {'rgb_psnr': 28.548999, 'rgb_ssim': 0.811089, 'lpips': 0.267085, 'lr_max_abs': 2.384185791015625e-07, 'params': 2356696, 'gflops': 308.5078, 'gpu': 'NVIDIA GeForce RTX 3090', 'ms_fp32_339x510': 78.418, 'ms_fp16_339x510': 50.007},
    'SPANV2': {'rgb_psnr': 27.989892, 'rgb_ssim': 0.796165, 'lpips': 0.29634, 'lr_max_abs': 0.050102829933166504, 'params': 140944, 'gflops': 18.3627, 'gpu': 'NVIDIA GeForce RTX 3090', 'ms_fp32_339x510': 9.12, 'ms_fp16_339x510': 6.987},
    'SPANV2 + NuSE': {'rgb_psnr': 28.044281, 'rgb_ssim': 0.797652, 'lpips': 0.288856, 'lr_max_abs': 2.384185791015625e-07, 'params': 139000, 'gflops': 18.1613, 'gpu': 'NVIDIA GeForce RTX 3090', 'ms_fp32_339x510': 9.659, 'ms_fp16_339x510': 7.892},
}
LABEL_OF = {"SPANV2": "SPANV2", "DISP": "DISP", "PLKSR-Tiny": "PLKSR-Tiny",
            "SPANV2 + NuSE": "SPANV2 + NuSE k=8 (cũ)", "DISP + NuSE": "DISP + NuSE k=8 (cũ)",
            "PLKSR-Tiny + NuSE": "PLKSR-Tiny + NuSE k=8 (cũ)"}
REPORTED_ROWS = []
if "average_pool" in METHODS:
    for key, r in OLD_REPORTED.items():
        label = LABEL_OF[key]
        if label in MODELS_EVAL["average_pool"] or ("(cũ)" in label and not SHOW_OLD_PLUGIN):
            continue                       # đã có checkpoint: dùng số đo lại trực tiếp
        name = label.replace(" (cũ)", "") + " [số cũ]"
        REPORT["average_pool"][("DIV2K-valid", name)] = {k: r[k] for k in ("rgb_psnr", "rgb_ssim", "lpips", "lr_max_abs")}
        REPORTED_ROWS.append({"model": name, **r})
for r in REPORTED_ROWS:
    print(f"{r['model']:30s} PSNR {r['rgb_psnr']:.4f}  SSIM {r['rgb_ssim']:.4f}  LPIPS {r['lpips']:.4f}  "
          f"params {r['params'] / 1e3:.1f}K  GFLOPs {r['gflops']:.2f} | latency cũ @339x510 trên {r['gpu']}: "
          f"fp32 {r['ms_fp32_339x510']} ms, fp16 {r['ms_fp16_339x510']} ms")
if not REPORTED_ROWS:
    print("Mọi baseline đều có checkpoint: dùng số đo lại trực tiếp.")

### Kiểm tra checkpoint cũ

PSNR đo lại bằng code của notebook này phải khớp với `comparison/summary.json` cũ (sai khác dưới 0.01 dB). Nếu không khớp thì lớp model dựng lại chưa đúng với code cũ.

In [ ]:
OLD_SUMMARY_KEY = {"SPANV2": "SPANV2", "DISP": "DISP", "PLKSR-Tiny": "PLKSR-Tiny",
                   "SPANV2 + NuSE k=8 (cũ)": "SPANV2 + NuSE", "DISP + NuSE k=8 (cũ)": "DISP + NuSE",
                   "PLKSR-Tiny + NuSE k=8 (cũ)": "PLKSR-Tiny + NuSE"}
for label, path in OLD.items():
    summary_path = Path(path).parent.parent / "comparison" / "summary.json"
    report = REPORT.get("average_pool", {}).get(("DIV2K-valid", label))
    if not summary_path.exists() or report is None:
        continue
    old = json.loads(summary_path.read_text())["reports"].get(OLD_SUMMARY_KEY[label])
    if old:
        delta = report["rgb_psnr"] - old["rgb_psnr"]
        print(f"{'OK ' if abs(delta) < 0.01 else 'LỆCH'} {label:28s} cũ {old['rgb_psnr']:.4f} | đo lại {report['rgb_psnr']:.4f} | {delta:+.4f} dB")

### Chất lượng theo độ chính xác số (fp32/fp16) và kiểu output (float/uint8), DIV2K valid

In [ ]:
QBC = []
for method in METHODS:
    for label, (model, k) in MODELS_EVAL[method].items():
        if "(cũ)" in label or NAME_OF(label) not in FULL[method]:
            continue
        for amp in ([False, True] if DEVICE.type == "cuda" else [False]):
            s, _ = L.evaluate(model, val_images, method, DEVICE, k=k, amp=amp)
            QBC += [{"lr": method, "model": label, "precision": "fp16" if amp else "fp32", "output": "float",
                     "psnr": s["rgb_psnr"], "ssim": s["rgb_ssim"]},
                    {"lr": method, "model": label, "precision": "fp16" if amp else "fp32", "output": "uint8",
                     "psnr": s["rgb_psnr_uint8"], "ssim": float("nan")}]
L.write_csv(COMPARE / "quality_by_config.csv", QBC)
print(len(QBC), "dòng ->", COMPARE / "quality_by_config.csv")

## Bước 6: chi phí và latency

- Tham số học được (buffer PCA cố định không tính).
- GFLOPs trên LR 256×256 (quy ước NTIRE ESR).
- Latency đo như notebook cũ: batch 1, CUDA events, các model đo xen kẽ qua nhiều vòng, fp32 và fp16.
- NuSE-E được tách thành predictor và decoder + data consistency.

In [ ]:
torch.backends.cudnn.benchmark = True
COST, LATENCY = [], []
for method in METHODS:
    for label, (model, k) in MODELS_EVAL[method].items():
        COST.append({"lr": method, "model": label, "params": L.count_params(model, k),
                     "gflops": L.gflops(model, FLOPS_SIZE, DEVICE, k)})
    if DEVICE.type == "cuda":
        for size in LATENCY_SIZES:
            x = torch.rand(1, 3, *size, device=DEVICE)
            fns = {}
            for label, (model, k) in MODELS_EVAL[method].items():
                if "(cũ)" in label or (k is not None and k != model.budgets[0]):
                    continue
                with torch.no_grad():
                    entries = L.stage_functions(NAME_OF(label), model, x)
                if k is not None and NAME_OF(label) not in FULL[method]:      # biến thể không chọn: chỉ K lớn nhất
                    entries = {key: fn for key, fn in entries.items() if key[0].endswith(f" K={model.budgets[-1]}")}
                fns.update(entries)
            for precision in LATENCY_PRECISIONS:
                times, memory = L.time_cuda(fns, precision, WARMUP, ITERS, ROUNDS)
                for (label, stage), values in times.items():
                    LATENCY.append({"lr": method, "device": torch.cuda.get_device_name(0), "lr_size": f"{size[0]}x{size[1]}",
                                    "precision": precision, "model": label, "stage": stage,
                                    "median_ms": statistics.median(values), "p90_ms": sorted(values)[int(0.9 * len(values))],
                                    "peak_mem_mb": memory[(label, stage)]})
    if CPU_LATENCY_SIZE:
        x = torch.rand(1, 3, *CPU_LATENCY_SIZE)
        for label, (model, k) in MODELS_EVAL[method].items():
            if "(cũ)" in label or NAME_OF(label) not in FULL[method]:
                continue
            cpu_model = __import__("copy").deepcopy(model).cpu()
            with torch.no_grad():
                values = L.time_cpu(lambda: cpu_model(x, k) if k else cpu_model(x), CPU_ITERS)
            LATENCY.append({"lr": method, "device": f"cpu ({torch.get_num_threads()} threads)",
                            "lr_size": f"{CPU_LATENCY_SIZE[0]}x{CPU_LATENCY_SIZE[1]}", "precision": "fp32",
                            "model": label, "stage": "total", "median_ms": statistics.median(values),
                            "p90_ms": sorted(values)[int(0.9 * len(values))], "peak_mem_mb": float("nan")})
for r in REPORTED_ROWS:
    COST.append({"lr": "average_pool", "model": r["model"], "params": r["params"], "gflops": r["gflops"]})
L.write_csv(COMPARE / "cost.csv", COST)
if LATENCY:
    L.write_csv(COMPARE / "latency.csv", LATENCY)
print(len(COST), "dòng chi phí,", len(LATENCY), "dòng latency")

## Bước 7: bảng tổng hợp

In [ ]:
from IPython.display import Markdown, display

def table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(map(str, r)) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

def fmt(v, d=4):
    return "—" if v is None or v != v else ("∞" if v == float("inf") else f"{v:.{d}f}")

def lat(method, label, size, precision, stage="total"):
    for r in LATENCY:
        if (r["lr"], r["model"], r["lr_size"], r["precision"], r["stage"]) == (method, label, size, precision, stage) \
                and not r["device"].startswith("cpu"):
            return r["median_ms"]
    return None

def cost(method, label):
    return next((c for c in COST if c["lr"] == method and c["model"] == label), None)

REF = f"{LATENCY_SIZES[-1][0]}x{LATENCY_SIZES[-1][1]}"
SUMMARY = {}
for method in METHODS:
    display(Markdown(f"### LR = {method}, DIV2K valid"))
    rows = []
    for (set_name, label), r in REPORT[method].items():
        if set_name != "DIV2K-valid":
            continue
        c = cost(method, label)
        rows.append([label, fmt(r["rgb_psnr"]), fmt(r["rgb_ssim"]), fmt(r.get("y_psnr_shave4")), fmt(r.get("y_ssim_shave4")),
                     fmt(r.get("lpips")), fmt(r.get("fid_patch256"), 2), f"{r['lr_max_abs']:.1e}",
                     f"{c['params'] / 1e3:.1f}K" if c else "—", fmt(c["gflops"], 2) if c else "—",
                     fmt(lat(method, label, REF, "fp32"), 2), fmt(lat(method, label, REF, "fp16"), 2)])
    table(["model", "PSNR ↑", "SSIM ↑", "Y-PSNR ↑", "Y-SSIM ↑", "LPIPS ↓", "FID ↓", "max|A(SR)−y|",
           "params", f"GFLOPs@{FLOPS_SIZE[0]}²", f"ms fp32 @{REF}", f"ms fp16 @{REF}"], rows)
    bench = [(s, lab, r) for (s, lab), r in REPORT[method].items() if s != "DIV2K-valid"]
    if bench:
        display(Markdown(f"**Benchmark (Y-PSNR / Y-SSIM, shave 4), LR = {method}**"))
        sets = sorted({s for s, _, _ in bench})
        labels = list(dict.fromkeys(lab for _, lab, _ in bench))
        table(["model", *sets], [[lab, *[next((f"{r['y_psnr_shave4']:.2f} / {r['y_ssim_shave4']:.4f}"
                                              for s2, l2, r in bench if s2 == s and l2 == lab), "—") for s in sets]]
                                 for lab in labels])
    SUMMARY[method] = {"reports": {f"{s} | {lab}": r for (s, lab), r in REPORT[method].items()},
                       "cost": [c for c in COST if c["lr"] == method], "architecture": ARCH[method],
                       "selected": SELECTED[method]}
    L.write_json(COMPARE / method / "summary.json", SUMMARY[method])

display(Markdown("### Các biến thể NuSE-E và ràng buộc cứng (sau khi train)"))
CONSTRAINT_ROWS = []
for method in METHODS:
    for name in NUSE:
        labels = [lab for (s, lab) in REPORT[method] if s == "DIV2K-valid" and NAME_OF(lab) == name]
        if not labels:
            continue
        arch = ARCH[method][name]
        ks = arch["budgets"]
        psnr = [REPORT[method][("DIV2K-valid", f"{name} K={k}")]["rgb_psnr"] for k in ks]
        params = [cost(method, f"{name} K={k}")["params"] for k in ks]
        ms = [lat(method, f"{name} K={k}", f"{FLOPS_SIZE[0]}x{FLOPS_SIZE[1]}", "fp32") for k in ks]
        ms_known = [v for v in ms if v is not None]
        consistency = max(REPORT[method][("DIV2K-valid", f"{name} K={k}")]["lr_max_abs"] for k in ks)
        checks = {"params ≤ trần": params[-1] <= PARAM_CAP,
                  "latency ≤ trần": (ms[-1] is None) or ms[-1] <= cap_ms,
                  "params tăng theo K": all(b > a for a, b in zip(params, params[1:])),
                  "latency tăng theo K": all(b > a for a, b in zip(ms_known, ms_known[1:])),
                  "PSNR tăng theo K": all(b > a for a, b in zip(psnr, psnr[1:])),
                  "A(SR)=y": consistency < CONSISTENCY_TOL}
        CONSTRAINT_ROWS.append({"lr": method, "model": name, "selected": name == SELECTED[method], "W": arch["width"],
                                "block": arch["block"], "N": sum(arch["blocks"]), "alpha": arch["alpha"], "K": str(ks),
                                "params_kmax": params[-1], "ms_kmax_fp32_256": ms[-1],
                                **{f"psnr_K{k}": p for k, p in zip(ks, psnr)}, "psnr_kmax": psnr[-1],
                                "lr_max_abs": consistency, **{f"ok: {k}": v for k, v in checks.items()}, "all_ok": all(checks.values())})
L.write_csv(COMPARE / "constraints.csv", CONSTRAINT_ROWS)
table(["LR", "model", "W", "block", "N", "alpha", "K", "params K max", "ms fp32 @256² K max", "PSNR theo K", "max|A(SR)−y|", "ràng buộc"],
      [[r["lr"], ("★ " if r["selected"] else "") + r["model"], r["W"], r["block"], r["N"], r["alpha"], r["K"], f"{r['params_kmax']:,}",
        fmt(r["ms_kmax_fp32_256"], 2), " / ".join(f"{r[k]:.3f}" for k in r if k.startswith("psnr_K")),
        f"{r['lr_max_abs']:.1e}", "đạt hết" if r["all_ok"] else "VI PHẠM: " + ", ".join(k[4:] for k in r if k.startswith("ok: ") and not r[k])]
       for r in CONSTRAINT_ROWS])
for r in CONSTRAINT_ROWS:
    if r["selected"] and not r["all_ok"]:
        print(f"CẢNH BÁO: bản được chọn {r['model']} / {r['lr']} vi phạm ràng buộc, xem cột cuối.")

def canonical(method, label):
    # NuSE-E có tập K khác nhau theo loại LR: so ở K nhỏ nhất và K lớn nhất; chỉ bản v2 và bản được chọn
    if not label.startswith("NuSE-E"):
        return label
    name = NAME_OF(label)
    if name not in ("NuSE-E", SELECTED[method]):
        return None
    k, budgets = int(label.split("=")[1]), ARCH[method][name]["budgets"]
    return f"{name} (K lớn nhất)" if k == budgets[-1] else f"{name} (K nhỏ nhất)" if k == budgets[0] else None

CROSS = {}
for m in METHODS:
    for (s, lab), r in REPORT[m].items():
        c = canonical(m, lab)
        if s == "DIV2K-valid" and c:
            CROSS.setdefault(c, {})[m] = (lab, r)
display(Markdown("### So sánh hai loại LR (DIV2K valid)"))
rows = []
for c, per in CROSS.items():
    row = [c]
    for m in METHODS:
        lab, r = per.get(m, (None, None))
        row.append("—" if r is None else f"{r['rgb_psnr']:.3f} / {r['rgb_ssim']:.4f} / {fmt(r.get('lpips'))} / {fmt(r.get('fid_patch256'), 1)}"
                   + (f" ({lab.split(' ')[1]})" if lab.startswith("NuSE-E") else ""))
    rows.append(row)
table(["model", *[f"{m}: PSNR / SSIM / LPIPS / FID" for m in METHODS]], rows)
if LATENCY:
    display(Markdown("### NuSE-E: phân rã latency (median ms, GPU)"))
    rows = []
    for m in METHODS:
        for name in dict.fromkeys(n for n in ("NuSE-E", SELECTED[m]) if n and n in ARCH[m]):
            for k in ARCH[m][name]["budgets"]:
                for size in [f"{h}x{w}" for h, w in LATENCY_SIZES]:
                    for p in LATENCY_PRECISIONS:
                        pr, dc, tt = (lat(m, f"{name} K={k}", size, p, s) for s in ("predictor", "decoder + consistency", "total"))
                        if tt:
                            rows.append([m, name, k, size, p, fmt(pr, 3), fmt(dc, 3), fmt(tt, 3), f"{dc / tt:.0%}"])
    table(["LR", "model", "K", "kích thước LR", "precision", "predictor", "decoder + consistency", "total", "tỉ lệ decoder"], rows)

## Bước 8: hình so sánh

In [ ]:
import numpy as np
from torch.nn import functional as F
COLORS = {"SPANV2": "#2a78d6", "DISP": "#eb6834", "PLKSR-Tiny": "#1baf7a", "Bicubic": "#8a8984"}

for method in METHODS:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, xkey, xlabel in [(axes[0], "lat", f"latency fp16 @LR {REF} (ms)"), (axes[1], "gflops", f"GFLOPs @LR {FLOPS_SIZE[0]}²")]:
        nuse = {}
        for (s, lab), r in REPORT[method].items():
            if s != "DIV2K-valid" or lab == "Bicubic":
                continue
            x = lat(method, lab, REF, "fp16") if xkey == "lat" else (cost(method, lab) or {}).get("gflops")
            if x is None:
                continue
            if lab.startswith("NuSE-E"):
                nuse.setdefault(NAME_OF(lab), []).append((x, r["rgb_psnr"], lab.split("=")[1]))
            else:
                base = lab.split(" +")[0]
                marker = "s" if "[số cũ]" in lab else "^" if "(cũ)" in lab or "NuSE k=8" in lab else "o"
                ax.scatter(x, r["rgb_psnr"], s=60, color=COLORS.get(base.replace(" [số cũ]", ""), "#6250d6"), marker=marker)
                ax.annotate(lab, (x, r["rgb_psnr"]), textcoords="offset points", xytext=(5, 4), fontsize=8)
        palette = ["#e34948", "#b3267a", "#8f4fc9", "#d98a1a", "#5c6b73", "#2d9a8f", "#a0522d", "#7a7a1a", "#1f4e8c", "#c71585"]
        for i, (name, pts) in enumerate(nuse.items()):
            pts.sort()
            main = name in ("NuSE-E", SELECTED[method])
            color = palette[i % len(palette)]
            ax.plot([p[0] for p in pts], [p[1] for p in pts], "-o" if main else ":o", color=color,
                    lw=1.8 if main else 1.0, ms=6 if main else 4, label=name + (" ★" if name == SELECTED[method] else ""))
            if main:
                for x, y, k in pts:
                    ax.annotate(f"K={k}", (x, y), textcoords="offset points", xytext=(4, -10), fontsize=8, color=color)
        ax.set(xlabel=xlabel, ylabel="PSNR DIV2K valid (dB)", title=f"LR {method}")
        if xkey == "gflops":
            ax.set_xscale("log")
        ax.grid(alpha=0.3); ax.legend(fontsize=8)
    plt.tight_layout(); fig.savefig(FIG / f"2_psnr_vs_cost_{method}.png", dpi=150); plt.show()

if any(not r["device"].startswith("cpu") for r in LATENCY):
    fig, axes = plt.subplots(1, len(METHODS), figsize=(7 * len(METHODS), 4.5), squeeze=False)
    for ax, method in zip(axes[0], METHODS):
        labels = [l for l in MODELS_EVAL[method] if "(cũ)" not in l and NAME_OF(l) in FULL[method]]
        pos = np.arange(len(labels))
        pred = [lat(method, l, REF, "fp16", "predictor") or lat(method, l, REF, "fp16") or 0 for l in labels]
        dec = [lat(method, l, REF, "fp16", "decoder + consistency") or 0 for l in labels]
        ax.bar(pos, pred, 0.6, label="mạng / predictor")
        ax.bar(pos, dec, 0.6, bottom=pred, label="decoder + consistency (NuSE-E)")
        ax.set_xticks(pos, labels, rotation=30, ha="right", fontsize=8)
        ax.set(ylabel="ms (median, fp16)", title=f"Latency @LR {REF}, LR {method}")
        ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=8)
    plt.tight_layout(); fig.savefig(FIG / "4_latency_breakdown.png", dpi=150); plt.show()

metrics = [("rgb_psnr", "PSNR ↑"), ("rgb_ssim", "SSIM ↑"), ("lpips", "LPIPS ↓"), ("fid_patch256", "FID ↓")]
labels = [c for c in CROSS if "(cũ)" not in c]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (key, name) in zip(axes, metrics):
    width = 0.8 / len(METHODS)
    for i, m in enumerate(METHODS):
        vals = [CROSS[c][m][1].get(key, np.nan) if m in CROSS[c] else np.nan for c in labels]
        ax.bar(np.arange(len(labels)) + i * width, vals, width, label=f"LR {m}")
    ax.set_xticks(np.arange(len(labels)) + width * (len(METHODS) - 1) / 2, labels, rotation=35, ha="right", fontsize=8)
    ax.set_title(name); ax.grid(axis="y", alpha=0.3)
    if key in ("rgb_psnr", "rgb_ssim"):
        finite = [v for m in METHODS for c in labels for v in [CROSS[c][m][1].get(key, np.nan) if m in CROSS[c] else np.nan] if v == v]
        if finite:
            ax.set_ylim(min(finite) - (0.5 if key == "rgb_psnr" else 0.02), max(finite) + (0.3 if key == "rgb_psnr" else 0.01))
axes[0].legend(fontsize=8)
plt.tight_layout(); fig.savefig(FIG / "3_avgpool_vs_bicubic.png", dpi=150); plt.show()

### So sánh từng ảnh và so sánh trực quan

NuSE-E ở K lớn nhất so với từng baseline. Hình trực quan: crop 192×192 ở vùng hai model khác nhau nhiều nhất, trên ảnh NuSE-E thắng nhiều nhất và thua nhiều nhất.

In [ ]:
CROP = 192
for method in METHODS:
    sel = SELECTED[method]
    if not sel:
        continue
    kmax = ARCH[method][sel]["budgets"][-1]
    ours_label = f"{sel} K={kmax}"
    ours = PER_IMAGE[method].get(("DIV2K-valid", ours_label))
    # so với baseline có checkpoint, với bản cắm NuSE cũ, và với bản v2 (nếu bản được chọn là biến thể)
    others = [l for l in MODELS_EVAL[method] if not l.startswith("NuSE-E")]
    if sel != "NuSE-E" and f"NuSE-E K={ARCH[method]['NuSE-E']['budgets'][-1]}" in MODELS_EVAL[method]:
        others.append(f"NuSE-E K={ARCH[method]['NuSE-E']['budgets'][-1]}")
    others = [l for l in others if ("DIV2K-valid", l) in PER_IMAGE[method]]
    if not ours or not others:
        continue
    fig, axes = plt.subplots(1, len(others), figsize=(5.5 * len(others), 4), squeeze=False)
    for ax, base in zip(axes[0], others):
        theirs = {r["name"]: r for r in PER_IMAGE[method][("DIV2K-valid", base)]}
        gain = [r["rgb_psnr"] - theirs[r["name"]]["rgb_psnr"] for r in ours]
        ax.hist(gain, bins=25, color="#2a78d6")
        ax.axvline(0, color="k", lw=1)
        ax.set(title=f"{ours_label} − {base}: thắng {sum(g > 0 for g in gain)}/{len(gain)}, "
                     f"trung vị {statistics.median(gain):+.3f} dB", xlabel="ΔPSNR (dB)", )
        ax.title.set_fontsize(9)
    plt.tight_layout(); fig.savefig(FIG / f"5_per_image_{method}.png", dpi=150); plt.show()

    base = others[0]
    theirs = {r["name"]: r for r in PER_IMAGE[method][("DIV2K-valid", base)]}
    ranked = sorted(ours, key=lambda r: r["rgb_psnr"] - theirs[r["name"]]["rgb_psnr"])
    for pick in (ranked[-1]["name"], ranked[0]["name"]):
        hr = L.as_float(val_images[[p.stem for p in VAL_FILES].index(pick)][0], DEVICE)
        lr = L.degrade(hr, method)
        with torch.no_grad():
            outs = {"Bicubic": F.interpolate(lr, scale_factor=4, mode="bicubic", align_corners=False)}
            for label, (model, k) in MODELS_EVAL[method].items():
                if label.startswith("NuSE-E") and label not in (ours_label, *others):
                    continue
                outs[label] = model(lr, k) if k else model(lr)
        diff = F.avg_pool2d((outs[ours_label] - outs[base]).abs().mean(1, keepdim=True), min(CROP, hr.shape[-2]), stride=CROP // 4)
        row, col = divmod(int(diff.flatten().argmax()), diff.shape[-1])
        top, left = row * (CROP // 4), col * (CROP // 4)
        fig, axes = plt.subplots(1, len(outs) + 1, figsize=(3.6 * (len(outs) + 1), 4))
        for ax, (title, img) in zip(axes, [*outs.items(), ("HR", hr)]):
            ax.imshow(img[0, :, top:top + CROP, left:left + CROP].clamp(0, 1).permute(1, 2, 0).cpu().numpy())
            ax.axis("off")
            ax.set_title(title if title == "HR" else f"{title}\n{L.psnr(img, hr).item():.2f} dB", fontsize=9)
        fig.suptitle(f"LR {method} | {pick}")
        plt.tight_layout(); fig.savefig(FIG / f"6_visual_{method}_{pick}.png", dpi=150); plt.show()

## Bước 9: đóng gói

Tải `notebook_outputs.zip` ở tab Output, hoặc dùng output của phiên này làm Input cho phiên sau (`RESUME_FROM`). Checkpoint mỗi 10 epoch không được đưa vào zip.

In [ ]:
import zipfile
archive = OUT / "notebook_outputs.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUT.rglob("*"):
        if path.is_file() and path != archive and "cache" not in path.parts and not path.name.startswith("epoch_"):
            zf.write(path, path.relative_to(OUT.parent))
print(archive, f"{archive.stat().st_size / 1e6:.1f} MB")